In [ ]:
# @title
# Colab-friendly: fast, low-memory inspection only (no training)
# Assumes OUTPUT_ROOT is already defined as a pathlib.Path and npz files exist.

import os
import numpy as np
from collections import Counter
from pathlib import Path

# Speed / determinism knobs (useful later too)
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"

MODEL_CONFIGS = {
    'dofa': {'native_dim': 768, 'input_channels': 2},
    'ssl4eo': {'native_dim': 384, 'input_channels': 2},
    'scalemae': {'native_dim': 1024, 'input_channels': 3},
    'prithvi': {'native_dim': 768, 'input_channels': 6},
    'saratrx': {'input_channels': 1, 'native_dim': 768},  # not loaded by your function, kept for reference
    'sarjepa': {'input_channels': 1, 'native_dim': 768},
    'sardet100k': {'input_channels': 3, 'native_dim': 2048},
    'dinov3': {'native_dim': 768, 'input_channels': 3},
}

OUTPUT_ROOT = Path('/content/drive/MyDrive/PhD Research/iclrFoundationExp')

# Your loader (unchanged) but keep it lightweight for inspection
def load_tier3_ensemble_embeddings(dataset_name, split):
    path = OUTPUT_ROOT / 'experiments/tier3_ensemble' / f'{dataset_name}_{split}_all_models_non_harmonized.npz'
    data = np.load(path, allow_pickle=False)

    embeddings_dict = {}
    for model in ['dofa', 'ssl4eo', 'scalemae', 'prithvi', 'sardet100k', 'sarjepa', 'dinov3']:
        if model == 'dinov3':
            path = OUTPUT_ROOT / 'experiments/tier3_ensemble' / f'{dataset_name}_{split}_all_models.npz'
            data = np.load(path, allow_pickle=False)

        emb = data[f'{model}_embeddings'].astype(np.float32)  # (N, D)

        # Normalize per split per model (as you do). For later experiments, consider
        # computing normalization stats on TRAIN only, but for inspection we keep your exact behavior.
        emb_mean = emb.mean(axis=0, keepdims=True)
        emb_std = emb.std(axis=0, keepdims=True)
        emb_std[emb_std < 1e-7] = 1.0
        embeddings_dict[model] = (emb - emb_mean) / emb_std

    labels = data['labels']  # could be strings or ints depending on how saved
    return embeddings_dict, labels

def _label_summary(y):
    y = np.asarray(y)
    # handle bytes labels if present
    if y.dtype.kind in ("S", "a"):
        y = np.array([t.decode("utf-8") for t in y], dtype=object)
    counts = Counter(y.tolist())
    return counts, len(y), len(counts)

def inspect_dataset(dataset_name, split):
    embs, y = load_tier3_ensemble_embeddings(dataset_name, split)

    counts, n, k = _label_summary(y)
    print(f"\n=== {dataset_name} | split={split} ===")
    print(f"N={n}  num_classes={k}")
    print("Class counts (top to bottom):")
    for cls, c in sorted(counts.items(), key=lambda x: (-x[1], str(x[0]))):
        print(f"  {cls}: {c}")

    # Embedding structure checks
    models = list(embs.keys())
    print("\nEmbeddings:")
    for m in models:
        X = embs[m]
        d_expected = MODEL_CONFIGS.get(m, {}).get("native_dim", None)
        msg = f"  {m:10s} shape={tuple(X.shape)} dtype={X.dtype}"
        if d_expected is not None:
            msg += f" expected_dim={d_expected}"
        # quick sanity stats (cheap)
        msg += f" mean={float(X.mean()):.3f} std={float(X.std()):.3f}"
        print(msg)

    # Consistency: all models should have same N
    ns = {m: embs[m].shape[0] for m in models}
    if len(set(ns.values())) != 1:
        print("WARNING: inconsistent N across models:", ns)

    # Print unique label values (truncated)
    uniq = np.unique(np.asarray(y))
    print(f"\nUnique labels sample ({min(30, len(uniq))}/{len(uniq)}): {uniq[:30]}")

# Run inspection for both datasets and splits
for dataset_name in ["opensarship", "fusarship"]:
    for split in ["train", "val", "test"]:
        inspect_dataset(dataset_name, split)


=== opensarship | split=train ===
N=12057  num_classes=6
Class counts (top to bottom):
  Cargo: 8484
  Tanker: 2920
  Dredging: 227
  Fishing: 222
  Passenger: 105
  Tug: 99

Embeddings:
  dofa       shape=(12057, 768) dtype=float32 expected_dim=768 mean=-0.000 std=1.000
  ssl4eo     shape=(12057, 384) dtype=float32 expected_dim=384 mean=-0.000 std=1.000
  scalemae   shape=(12057, 1024) dtype=float32 expected_dim=1024 mean=0.000 std=1.000
  prithvi    shape=(12057, 768) dtype=float32 expected_dim=768 mean=-0.000 std=1.000
  sardet100k shape=(12057, 2048) dtype=float32 expected_dim=2048 mean=-0.000 std=1.000
  sarjepa    shape=(12057, 768) dtype=float32 expected_dim=768 mean=-0.000 std=1.000
  dinov3     shape=(12057, 768) dtype=float32 expected_dim=768 mean=0.000 std=1.000

Unique labels sample (6/6): ['Cargo' 'Dredging' 'Fishing' 'Passenger' 'Tanker' 'Tug']

=== opensarship | split=val ===
N=1505  num_classes=6
Class counts (top to bottom):
  Cargo: 1060
  Tanker: 365
  Dredging: 28


In [ ]:
# @title
import numpy as np

def inspect_npz(path, max_arrays=60):
    print(f"\n=== {path} ===")
    z = np.load(path, allow_pickle=False)
    keys = sorted(list(z.keys()))
    print(f"num_keys={len(keys)}")
    for i, k in enumerate(keys[:max_arrays]):
        arr = z[k]
        # arr can be ndarray; show minimal info
        shape = getattr(arr, "shape", None)
        dtype = getattr(arr, "dtype", None)
        print(f"  {i:02d} {k:40s} shape={shape} dtype={dtype}")
    if len(keys) > max_arrays:
        print(f"  ... ({len(keys)-max_arrays} more keys not shown)")
    z.close()

def paths_for(dataset_name, split):
    p1 = OUTPUT_ROOT / "experiments/tier3_ensemble" / f"{dataset_name}_{split}_all_models_non_harmonized.npz"
    p2 = OUTPUT_ROOT / "experiments/tier3_ensemble" / f"{dataset_name}_{split}_all_models.npz"  # used for dinov3 in your loader
    return p1, p2

for dataset_name in ["opensarship", "fusarship"]:
    for split in ["train", "val", "test"]:
        p_nonharm, p_all = paths_for(dataset_name, split)
        inspect_npz(p_nonharm)
        inspect_npz(p_all)


=== /content/drive/MyDrive/PhD Research/iclrFoundationExp/experiments/tier3_ensemble/opensarship_train_all_models_non_harmonized.npz ===
num_keys=9
  00 dofa_embeddings                          shape=(12057, 768) dtype=float32
  01 groups                                   shape=(12057,) dtype=<U8
  02 labels                                   shape=(12057,) dtype=<U9
  03 paths                                    shape=(12057,) dtype=<U127
  04 prithvi_embeddings                       shape=(12057, 768) dtype=float32
  05 sardet100k_embeddings                    shape=(12057, 2048) dtype=float32
  06 sarjepa_embeddings                       shape=(12057, 768) dtype=float32
  07 scalemae_embeddings                      shape=(12057, 1024) dtype=float32
  08 ssl4eo_embeddings                        shape=(12057, 384) dtype=float32

=== /content/drive/MyDrive/PhD Research/iclrFoundationExp/experiments/tier3_ensemble/opensarship_train_all_models.npz ===
num_keys=4
  00 dinov3_embeddings    

In [ ]:
### First Cell Always run
from pathlib import Path

OUTPUT_ROOT = Path("/content/drive/MyDrive/PhD Research/iclrFoundationExp")

## Preparing data

In [ ]:
import json
from pathlib import Path
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
# Step 1: Prepare experiment-ready data for ICML EIML study
# - No additional normalization (uses stored embeddings as-is)
# - Builds 5 stratified random splits for paper classes: Cargo, Tanker, Fishing
# - Creates an organized directory structure under OUTPUT_ROOT / "icml_eiml"
# - Saves reusable index files and metadata only (no heavy duplicated embeddings)

# =========================
# User-configurable settings
# =========================
# IMPORTANT: OUTPUT_ROOT must already exist in your Colab environment.
# Example:
OUTPUT_ROOT = Path("/content/drive/MyDrive/PhD Research/iclrFoundationExp")
assert "OUTPUT_ROOT" in globals(), "Please define OUTPUT_ROOT (pathlib.Path) before running."

DATASETS = ["opensarship", "fusarship"]
PAPER_CLASSES = ["Cargo", "Tanker", "Fishing"]
SPLIT_IDS = [0, 1, 2, 3, 4]
SPLIT_SEEDS = [101, 202, 303, 404, 505]   # one seed per split_id
RATIOS = (0.8, 0.1, 0.1)                   # train, val, test
MIN_PER_CLASS_FOR_VAL_TEST = 2             # safety
MODELS_NON_DINO = ["dofa", "ssl4eo", "scalemae", "prithvi", "sardet100k", "sarjepa"]
MODEL_DINO = "dinov3"

# =========================
# Paths
# =========================
EXP_ROOT = OUTPUT_ROOT / "icml_eiml"
DIRS = {
    "meta": EXP_ROOT / "meta",
    "indices": EXP_ROOT / "indices",
    "logs": EXP_ROOT / "logs",
    "checkpoints": EXP_ROOT / "checkpoints",
    "predictions": EXP_ROOT / "predictions",
    "metrics": EXP_ROOT / "metrics",
    "analysis": EXP_ROOT / "analysis",
    "cache": EXP_ROOT / "cache",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

Utilities

In [ ]:
# =========================
# Helpers
# =========================
def npz_paths(dataset_name: str, split: str):
    base = OUTPUT_ROOT / "experiments" / "tier3_ensemble"
    p_non_h = base / f"{dataset_name}_{split}_all_models_non_harmonized.npz"
    p_all = base / f"{dataset_name}_{split}_all_models.npz"
    return p_non_h, p_all

def load_labels_paths_for_original_split(dataset_name: str, split: str):
    """
    Loads labels + paths from non-harmonized file.
    Uses only metadata arrays, no embedding arrays.
    """
    p_non_h, _ = npz_paths(dataset_name, split)
    if not p_non_h.exists():
        raise FileNotFoundError(f"Missing file: {p_non_h}")

    with np.load(p_non_h, allow_pickle=False) as z:
        labels = z["labels"].astype(str)
        paths = z["paths"].astype(str)
    return labels, paths

def combine_original_splits(dataset_name: str):
    """
    Combine train/val/test pools into one unified pool for fresh 5 random splits.
    Keeps a pointer to source original split/index for traceability.
    """
    all_labels, all_paths = [], []
    src_split, src_idx = [], []

    offset = 0
    for split in ["train", "val", "test"]:
        y, p = load_labels_paths_for_original_split(dataset_name, split)
        n = len(y)
        all_labels.append(y)
        all_paths.append(p)
        src_split.append(np.array([split] * n, dtype=object))
        src_idx.append(np.arange(n, dtype=np.int32))

    labels = np.concatenate(all_labels, axis=0).astype(str)
    paths = np.concatenate(all_paths, axis=0).astype(str)
    original_split = np.concatenate(src_split, axis=0).astype(str)
    original_idx = np.concatenate(src_idx, axis=0).astype(np.int32)

    return {
        "labels": labels,
        "paths": paths,
        "original_split": original_split,
        "original_idx": original_idx,
    }

def filter_to_paper_classes(pool_dict, paper_classes):
    labels = pool_dict["labels"]
    mask = np.isin(labels, np.array(paper_classes, dtype=object))
    out = {}
    for k, v in pool_dict.items():
        out[k] = v[mask]
    return out

def stratified_three_way_split(y_str, seed, ratios=(0.8, 0.1, 0.1)):
    """
    Returns train_idx, val_idx, test_idx on local array index space [0..N-1].
    """
    r_train, r_val, r_test = ratios
    assert abs((r_train + r_val + r_test) - 1.0) < 1e-9, "Ratios must sum to 1."

    n = len(y_str)
    idx_all = np.arange(n, dtype=np.int32)

    # First split: train vs temp
    temp_size = r_val + r_test
    idx_train, idx_temp, y_train, y_temp = train_test_split(
        idx_all, y_str,
        test_size=temp_size,
        random_state=seed,
        stratify=y_str
    )

    # Second split: val vs test from temp
    # val fraction within temp = r_val / (r_val+r_test)
    val_frac_within_temp = r_val / (r_val + r_test)
    idx_val, idx_test, _, _ = train_test_split(
        idx_temp, y_temp,
        test_size=(1.0 - val_frac_within_temp),
        random_state=seed + 1,
        stratify=y_temp
    )

    return np.sort(idx_train), np.sort(idx_val), np.sort(idx_test)

def count_by_class(y):
    cls, cnt = np.unique(y, return_counts=True)
    return {str(c): int(n) for c, n in zip(cls, cnt)}

def assert_class_minimum(y_split, min_per_class=2):
    counts = count_by_class(y_split)
    for c in PAPER_CLASSES:
        if counts.get(c, 0) < min_per_class:
            raise ValueError(f"Class '{c}' has only {counts.get(c, 0)} samples in a split, below minimum {min_per_class}.")

In [ ]:
# =========================
# Build metadata manifest
# =========================
manifest = {
    "experiment_name": "icml_eiml",
    "paper_classes": PAPER_CLASSES,
    "datasets": DATASETS,
    "num_random_splits": len(SPLIT_IDS),
    "split_ids": SPLIT_IDS,
    "split_seeds": SPLIT_SEEDS,
    "ratios": {"train": RATIOS[0], "val": RATIOS[1], "test": RATIOS[2]},
    "normalization_policy": "no_additional_normalization",
    "notes": "Indices are created from combined original train/val/test pools, filtered to paper classes.",
}
(DIRS["meta"] / "manifest.json").write_text(json.dumps(manifest, indent=2))

# Fixed label mapping for consistency everywhere
label_to_id = {"Cargo": 0, "Tanker": 1, "Fishing": 2}
(DIRS["meta"] / "label_to_id.json").write_text(json.dumps(label_to_id, indent=2))

# =========================
# Main processing per dataset
# =========================
summary = {"datasets": {}}

for dataset_name in DATASETS:
    ds_dir = DIRS["indices"] / dataset_name
    ds_dir.mkdir(parents=True, exist_ok=True)

    # 1) Build unified pool from original splits
    pool = combine_original_splits(dataset_name)
    pool_f = filter_to_paper_classes(pool, PAPER_CLASSES)

    y_all = pool_f["labels"].astype(str)
    p_all = pool_f["paths"].astype(str)
    src_split_all = pool_f["original_split"].astype(str)
    src_idx_all = pool_f["original_idx"].astype(np.int32)

    n_total = len(y_all)
    if n_total == 0:
        raise ValueError(f"No samples found for paper classes in dataset={dataset_name}")

    # dataset-level metadata
    ds_meta = {
        "dataset_name": dataset_name,
        "total_filtered_samples": int(n_total),
        "class_counts_total": count_by_class(y_all),
    }

    # Save filtered pool references (small metadata, no embeddings)
    # This allows traceability and debugging.
    np.savez_compressed(
        ds_dir / "filtered_pool_refs.npz",
        labels=y_all,
        paths=p_all,
        original_split=src_split_all,
        original_idx=src_idx_all,
        label_ids=np.array([label_to_id[x] for x in y_all], dtype=np.int16),
    )

    # 2) Create 5 random stratified splits
    split_summaries = {}
    for split_id, split_seed in zip(SPLIT_IDS, SPLIT_SEEDS):
        tr_idx, va_idx, te_idx = stratified_three_way_split(y_all, seed=int(split_seed), ratios=RATIOS)

        y_tr, y_va, y_te = y_all[tr_idx], y_all[va_idx], y_all[te_idx]

        # Safety checks for tiny classes
        assert_class_minimum(y_tr, MIN_PER_CLASS_FOR_VAL_TEST)
        assert_class_minimum(y_va, MIN_PER_CLASS_FOR_VAL_TEST)
        assert_class_minimum(y_te, MIN_PER_CLASS_FOR_VAL_TEST)

        # Save split indices + labels for direct use in later stages
        np.savez_compressed(
            ds_dir / f"split_{split_id}.npz",
            split_id=np.int16(split_id),
            split_seed=np.int32(split_seed),
            train_idx=tr_idx.astype(np.int32),
            val_idx=va_idx.astype(np.int32),
            test_idx=te_idx.astype(np.int32),
            train_labels=y_tr.astype(str),
            val_labels=y_va.astype(str),
            test_labels=y_te.astype(str),
            train_label_ids=np.array([label_to_id[x] for x in y_tr], dtype=np.int16),
            val_label_ids=np.array([label_to_id[x] for x in y_va], dtype=np.int16),
            test_label_ids=np.array([label_to_id[x] for x in y_te], dtype=np.int16),
        )

        split_summaries[f"split_{split_id}"] = {
            "seed": int(split_seed),
            "n_train": int(len(tr_idx)),
            "n_val": int(len(va_idx)),
            "n_test": int(len(te_idx)),
            "class_counts_train": count_by_class(y_tr),
            "class_counts_val": count_by_class(y_va),
            "class_counts_test": count_by_class(y_te),
        }

    ds_meta["splits"] = split_summaries
    summary["datasets"][dataset_name] = ds_meta

# =========================
# Save and print summary
# =========================
summary_path = DIRS["meta"] / "step1_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print("Step 1 complete.")
print(f"Experiment root: {EXP_ROOT}")
print(f"Manifest: {DIRS['meta'] / 'manifest.json'}")
print(f"Label mapping: {DIRS['meta'] / 'label_to_id.json'}")
print(f"Summary: {summary_path}")

for ds in DATASETS:
    info = summary["datasets"][ds]
    print(f"\n[{ds}] total_filtered_samples={info['total_filtered_samples']}, class_counts_total={info['class_counts_total']}")
    for s in SPLIT_IDS:
        x = info["splits"][f"split_{s}"]
        print(
            f"  split_{s} seed={x['seed']} "
            f"| n(train/val/test)=({x['n_train']}/{x['n_val']}/{x['n_test']}) "
            f"| train_counts={x['class_counts_train']}"
        )

Step 1 complete.
Experiment root: /content/drive/MyDrive/PhD Research/iclrFoundationExp/icml_eiml
Manifest: /content/drive/MyDrive/PhD Research/iclrFoundationExp/icml_eiml/meta/manifest.json
Label mapping: /content/drive/MyDrive/PhD Research/iclrFoundationExp/icml_eiml/meta/label_to_id.json
Summary: /content/drive/MyDrive/PhD Research/iclrFoundationExp/icml_eiml/meta/step1_summary.json

[opensarship] total_filtered_samples=14534, class_counts_total={'Cargo': 10606, 'Fishing': 278, 'Tanker': 3650}
  split_0 seed=101 | n(train/val/test)=(11627/1453/1454) | train_counts={'Cargo': 8485, 'Fishing': 222, 'Tanker': 2920}
  split_1 seed=202 | n(train/val/test)=(11627/1453/1454) | train_counts={'Cargo': 8485, 'Fishing': 222, 'Tanker': 2920}
  split_2 seed=303 | n(train/val/test)=(11627/1453/1454) | train_counts={'Cargo': 8485, 'Fishing': 222, 'Tanker': 2920}
  split_3 seed=404 | n(train/val/test)=(11627/1453/1454) | train_counts={'Cargo': 8485, 'Fishing': 222, 'Tanker': 2920}
  split_4 seed=505

In [ ]:
import json
import numpy as np
from pathlib import Path

assert "OUTPUT_ROOT" in globals(), "Please define OUTPUT_ROOT first."

EXP_ROOT = OUTPUT_ROOT / "icml_eiml"
META_DIR = EXP_ROOT / "meta"
IDX_DIR = EXP_ROOT / "indices"
DATASETS = ["opensarship", "fusarship"]

def load_json(p):
    with open(p, "r") as f:
        return json.load(f)

def counts(y):
    u, c = np.unique(y, return_counts=True)
    return {str(k): int(v) for k, v in zip(u, c)}

def verify_dataset(ds, manifest, summary):
    print(f"\n=== VERIFY DATASET: {ds} ===")
    ds_dir = IDX_DIR / ds
    pool = np.load(ds_dir / "filtered_pool_refs.npz", allow_pickle=False)
    y_all = pool["labels"].astype(str)
    paths_all = pool["paths"].astype(str)
    n_all = len(y_all)

    print(f"Pool size: {n_all}")
    print(f"Pool class counts: {counts(y_all)}")

    sum_ds = summary["datasets"][ds]
    assert n_all == sum_ds["total_filtered_samples"]

    split_paths_sets = []
    for split_id in manifest["split_ids"]:
        z = np.load(ds_dir / f"split_{split_id}.npz", allow_pickle=False)
        tr, va, te = z["train_idx"], z["val_idx"], z["test_idx"]
        y_tr, y_va, y_te = z["train_labels"].astype(str), z["val_labels"].astype(str), z["test_labels"].astype(str)

        # bounds
        assert tr.min() >= 0 and tr.max() < n_all
        assert va.min() >= 0 and va.max() < n_all
        assert te.min() >= 0 and te.max() < n_all

        # disjoint
        assert len(set(tr).intersection(set(va))) == 0
        assert len(set(tr).intersection(set(te))) == 0
        assert len(set(va).intersection(set(te))) == 0

        # full cover
        union = np.concatenate([tr, va, te])
        assert len(union) == n_all
        assert len(np.unique(union)) == n_all

        # labels match source
        assert np.array_equal(y_all[tr], y_tr)
        assert np.array_equal(y_all[va], y_va)
        assert np.array_equal(y_all[te], y_te)

        split_paths_sets.append(set(paths_all[tr].tolist()))
        print(f"split_{split_id}: OK | sizes {len(tr)}/{len(va)}/{len(te)} | train_counts={counts(y_tr)}")

    base = split_paths_sets[0]
    for i in range(1, len(split_paths_sets)):
        cur = split_paths_sets[i]
        jac = len(base & cur) / len(base | cur)
        print(f"train Jaccard(split_0, split_{i}) = {jac:.4f} (should be < 1.0)")

manifest = load_json(META_DIR / "manifest.json")
summary = load_json(META_DIR / "step1_summary.json")
lmap = load_json(META_DIR / "label_to_id.json")

print("=== GLOBAL CHECKS ===")
print("Label map:", lmap)
assert lmap == {"Cargo": 0, "Tanker": 1, "Fishing": 2}

for ds in DATASETS:
    verify_dataset(ds, manifest, summary)

print("\nALL STEP 1 CHECKS PASSED.")

=== GLOBAL CHECKS ===
Label map: {'Cargo': 0, 'Tanker': 1, 'Fishing': 2}

=== VERIFY DATASET: opensarship ===
Pool size: 14534
Pool class counts: {'Cargo': 10606, 'Fishing': 278, 'Tanker': 3650}
split_0: OK | sizes 11627/1453/1454 | train_counts={'Cargo': 8485, 'Fishing': 222, 'Tanker': 2920}
split_1: OK | sizes 11627/1453/1454 | train_counts={'Cargo': 8485, 'Fishing': 222, 'Tanker': 2920}
split_2: OK | sizes 11627/1453/1454 | train_counts={'Cargo': 8485, 'Fishing': 222, 'Tanker': 2920}
split_3: OK | sizes 11627/1453/1454 | train_counts={'Cargo': 8485, 'Fishing': 222, 'Tanker': 2920}
split_4: OK | sizes 11627/1453/1454 | train_counts={'Cargo': 8485, 'Fishing': 222, 'Tanker': 2920}
train Jaccard(split_0, split_1) = 0.6647 (should be < 1.0)
train Jaccard(split_0, split_2) = 0.6681 (should be < 1.0)
train Jaccard(split_0, split_3) = 0.6666 (should be < 1.0)
train Jaccard(split_0, split_4) = 0.6678 (should be < 1.0)

=== VERIFY DATASET: fusarship ===
Pool size: 2626
Pool class counts: {'Ca

## Experiments

### Common

In [ ]:
import os
import gc
import json
import time
import random
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    roc_auc_score, average_precision_score
)

In [ ]:
# =========================
# Step 2 - Section A (Shared Setup)
# =========================
# What this section does:
# - Imports and global config
# - Reproducibility helpers
# - Paths and run keys
# - Data loading helpers (no extra normalization)
# - Train/val/test tensor builders from Step 1 indices
# - Model head, training loop, evaluation helpers
# - Save/load utilities for checkpoints, predictions, metrics
#
# Notes:
# - This section does NOT launch ID/CD/UC runs yet.
# - DINOv3 is removed (6 models only).
# - Keep this cell executed once, then run Section B/C/D cells.

# -------------------------
# Required root path
# -------------------------
assert "OUTPUT_ROOT" in globals(), "Please define OUTPUT_ROOT as pathlib.Path before running."
OUTPUT_ROOT = Path(OUTPUT_ROOT)

# -------------------------
# Experiment constants
# -------------------------
EXP_ROOT = OUTPUT_ROOT / "icml_eiml"
META_DIR = EXP_ROOT / "meta"
IDX_DIR = EXP_ROOT / "indices"
CKPT_DIR = EXP_ROOT / "checkpoints"
PRED_DIR = EXP_ROOT / "predictions"
METR_DIR = EXP_ROOT / "metrics"
LOG_DIR = EXP_ROOT / "logs"

for d in [CKPT_DIR, PRED_DIR, METR_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DATASETS = ["opensarship", "fusarship"]
MODELS = ["dofa", "ssl4eo", "scalemae", "prithvi", "sardet100k", "sarjepa"]  # no dinov3
CLASS_ORDER = ["Cargo", "Tanker", "Fishing"]
LABEL_TO_ID = {"Cargo": 0, "Tanker": 1, "Fishing": 2}
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}

SPLIT_IDS = [0, 1, 2, 3, 4] # full
SEEDS = [42, 123, 456, 789, 999]  # full
# SPLIT_IDS = [0, 1, 2] # small
# SEEDS = [42, 123, 456] # small

MODEL_DIMS = {
    "dofa": 768,
    "ssl4eo": 384,
    "scalemae": 1024,
    "prithvi": 768,
    "sardet100k": 2048,
    "sarjepa": 768,
}

TRAIN_CFG = {
    "epochs": 100,
    "batch_size": 512,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "patience": 20,
    "min_delta": 1e-4,
    "num_workers": 2,
    "pin_memory": True,
    "dropout_p": 0.2,   # used for MC-dropout baseline
}
# TRAIN_CFG = {
#     "epochs": 60,
#     "batch_size": 1024,
#     "lr": 2e-3,
#     "weight_decay": 1e-4,
#     "patience": 15,
#     "min_delta": 1e-4,
#     "num_workers": 4,
#     "pin_memory": True,
#     "dropout_p": 0.2,
# }

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using DEVICE = {DEVICE}")

# -------------------------
# Performance knobs
# -------------------------
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# -------------------------
# Reproducibility
# -------------------------
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# -------------------------
# Basic file path helpers
# -------------------------
def tier3_paths(dataset_name: str, split: str) -> Path:
    # No dinov3 here, so only non-harmonized file is needed.
    p = OUTPUT_ROOT / "experiments" / "tier3_ensemble" / f"{dataset_name}_{split}_all_models_non_harmonized.npz"
    if not p.exists():
        raise FileNotFoundError(f"Missing tier3 file: {p}")
    return p

def split_index_file(dataset_name: str, split_id: int) -> Path:
    p = IDX_DIR / dataset_name / f"split_{split_id}.npz"
    if not p.exists():
        raise FileNotFoundError(f"Missing split index file: {p}")
    return p

def pool_ref_file(dataset_name: str) -> Path:
    p = IDX_DIR / dataset_name / "filtered_pool_refs.npz"
    if not p.exists():
        raise FileNotFoundError(f"Missing filtered pool refs: {p}")
    return p

# -------------------------
# Load original embeddings+metadata for one dataset split (train/val/test)
# -------------------------
def load_original_split(dataset_name: str, split: str, model_name: str):
    p = tier3_paths(dataset_name, split)
    with np.load(p, allow_pickle=False) as z:
        X = z[f"{model_name}_embeddings"].astype(np.float32)  # no extra normalization
        y = z["labels"].astype(str)
        paths = z["paths"].astype(str)
    return X, y, paths

# -------------------------
# Build a unified filtered pool for one dataset + one model
# aligned with Step 1 filtered_pool_refs order
# -------------------------
def build_filtered_pool_embeddings(dataset_name: str, model_name: str):
    # Load the Step-1 filtered pool refs (labels/paths/original split+idx)
    pf = np.load(pool_ref_file(dataset_name), allow_pickle=False)
    labels_f = pf["labels"].astype(str)
    paths_f = pf["paths"].astype(str)
    orig_split_f = pf["original_split"].astype(str)
    orig_idx_f = pf["original_idx"].astype(np.int32)
    label_ids_f = pf["label_ids"].astype(np.int16)

    # Load all original splits once
    x_train, y_train, p_train = load_original_split(dataset_name, "train", model_name)
    x_val, y_val, p_val = load_original_split(dataset_name, "val", model_name)
    x_test, y_test, p_test = load_original_split(dataset_name, "test", model_name)

    # Collect for indexed lookup by (split, idx)
    X_by_split = {"train": x_train, "val": x_val, "test": x_test}
    Y_by_split = {"train": y_train, "val": y_val, "test": y_test}
    P_by_split = {"train": p_train, "val": p_val, "test": p_test}

    # Rebuild filtered pool embeddings in exact Step-1 order
    X_f = np.empty((len(labels_f), MODEL_DIMS[model_name]), dtype=np.float32)

    for i in range(len(labels_f)):
        s = orig_split_f[i]
        j = int(orig_idx_f[i])
        X_f[i] = X_by_split[s][j]

        # strict consistency checks
        if Y_by_split[s][j] != labels_f[i]:
            raise ValueError(f"Label mismatch at filtered index {i} ({dataset_name}, {model_name}).")
        if P_by_split[s][j] != paths_f[i]:
            raise ValueError(f"Path mismatch at filtered index {i} ({dataset_name}, {model_name}).")

    return {
        "X": X_f,                 # float32 [N, D]
        "y_str": labels_f,        # str [N]
        "y_id": label_ids_f,      # int16 [N]
        "paths": paths_f,         # str [N]
    }

# -------------------------
# Load split indices and slice a filtered pool
# -------------------------
def load_split_indices(dataset_name: str, split_id: int):
    z = np.load(split_index_file(dataset_name, split_id), allow_pickle=False)
    return {
        "train_idx": z["train_idx"].astype(np.int32),
        "val_idx": z["val_idx"].astype(np.int32),
        "test_idx": z["test_idx"].astype(np.int32),
    }

def slice_by_indices(pool_dict, idx: np.ndarray):
    return {
        "X": pool_dict["X"][idx],
        "y_str": pool_dict["y_str"][idx],
        "y_id": pool_dict["y_id"][idx].astype(np.int64),
        "paths": pool_dict["paths"][idx],
    }

# -------------------------
# Torch dataset
# -------------------------
class EmbDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.from_numpy(X)                  # float32
        self.y = torch.from_numpy(y.astype(np.int64)) # int64
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

def make_loader(X, y, batch_size, shuffle, num_workers=2, pin_memory=True):
    ds = EmbDataset(X, y)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False
    )

# -------------------------
# Model head
# -------------------------
class MLPHead(nn.Module):
    def __init__(self, in_dim: int, n_classes: int = 3, dropout_p: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(512, n_classes)
        )
    def forward(self, x):
        return self.net(x)

# -------------------------
# Utility: run forward
# -------------------------
@torch.no_grad()
def predict_logits(model, loader, device):
    model.eval()
    all_logits, all_targets = [], []
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        logits = model(xb).cpu().numpy()
        all_logits.append(logits)
        all_targets.append(yb.numpy())
    return np.concatenate(all_logits, axis=0), np.concatenate(all_targets, axis=0)

def softmax_np(logits):
    z = logits - logits.max(axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / ez.sum(axis=1, keepdims=True)

# -------------------------
# Baseline uncertainty scores
# -------------------------
@torch.no_grad()
def mc_dropout_probs(model, loader, device, mc_samples=10):
    # Enable dropout during inference
    model.train()
    probs_mc = []
    for _ in range(mc_samples):
        logits, _ = predict_logits(model, loader, device)
        probs_mc.append(softmax_np(logits))
    probs_mc = np.stack(probs_mc, axis=0)      # [T, N, C]
    mean_probs = probs_mc.mean(axis=0)         # [N, C]
    var_probs = probs_mc.var(axis=0).mean(axis=1)  # [N] scalar uncertainty
    return mean_probs, var_probs

# -------------------------
# Prototype-based uncertainty (class/domain)
# -------------------------
def fit_prototypes(X_train: np.ndarray, y_train: np.ndarray):
    # X_train: [N, D], y_train int labels
    mu_domain = X_train.mean(axis=0, keepdims=True)   # [1, D]
    class_mus = []
    for c in range(3):
        Xc = X_train[y_train == c]
        if len(Xc) == 0:
            raise ValueError(f"No samples for class {c} in training set.")
        class_mus.append(Xc.mean(axis=0))
    class_mus = np.stack(class_mus, axis=0)          # [C, D]
    return class_mus.astype(np.float32), mu_domain.astype(np.float32)

def score_class_domain_uncertainty(X: np.ndarray, class_mus: np.ndarray, mu_domain: np.ndarray):
    # class distance: min euclidean distance to class prototype
    # domain distance: euclidean distance to global domain center
    # both lower is more typical
    # return uncertainty as distances (higher means more uncertain)
    # X: [N, D], class_mus: [C, D], mu_domain: [1, D]
    # class
    d_class = np.sqrt(((X[:, None, :] - class_mus[None, :, :]) ** 2).sum(axis=2))  # [N,C]
    s_class = d_class.min(axis=1)                                                   # [N]
    # domain
    d_domain = np.sqrt(((X - mu_domain) ** 2).sum(axis=1))                          # [N]
    s_domain = d_domain
    return s_class.astype(np.float32), s_domain.astype(np.float32)

# -------------------------
# Training and early stopping
# -------------------------
def train_one_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: str,
    cfg: dict,
):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_state = None
    best_val_loss = float("inf")
    best_epoch = -1
    bad_epochs = 0
    history = []

    for epoch in range(cfg["epochs"]):
        # train
        model.train()
        tr_loss_sum, tr_n = 0.0, 0
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            bs = xb.size(0)
            tr_loss_sum += loss.item() * bs
            tr_n += bs

        train_loss = tr_loss_sum / max(tr_n, 1)

        # val
        model.eval()
        va_loss_sum, va_n = 0.0, 0
        all_logits, all_targets = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)
                logits = model(xb)
                loss = criterion(logits, yb)

                bs = xb.size(0)
                va_loss_sum += loss.item() * bs
                va_n += bs

                all_logits.append(logits.cpu().numpy())
                all_targets.append(yb.cpu().numpy())

        val_loss = va_loss_sum / max(va_n, 1)
        val_logits = np.concatenate(all_logits, axis=0)
        val_targets = np.concatenate(all_targets, axis=0)
        val_pred = val_logits.argmax(axis=1)
        val_acc = accuracy_score(val_targets, val_pred)

        history.append({
            "epoch": epoch + 1,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "val_acc": float(val_acc),
        })

        improved = (best_val_loss - val_loss) > cfg["min_delta"]
        if improved:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            bad_epochs = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= cfg["patience"]:
                break

    # restore best state
    if best_state is not None:
        model.load_state_dict(best_state)

    return model, {
        "best_val_loss": float(best_val_loss),
        "best_epoch": int(best_epoch),
        "history": history,
    }

# -------------------------
# Metrics helpers
# -------------------------
def classwise_accuracy(y_true, y_pred):
    out = {}
    for cid in [0, 1, 2]:
        mask = (y_true == cid)
        key = ID_TO_LABEL[cid]
        out[key] = float((y_pred[mask] == y_true[mask]).mean()) if mask.sum() > 0 else np.nan
    return out

def compute_basic_metrics(y_true, y_pred, probs):
    m = {}
    m["accuracy"] = float(accuracy_score(y_true, y_pred))
    m["balanced_accuracy"] = float(balanced_accuracy_score(y_true, y_pred))
    m["macro_f1"] = float(f1_score(y_true, y_pred, average="macro"))
    m["classwise_accuracy"] = classwise_accuracy(y_true, y_pred)
    m["mean_max_softmax"] = float(probs.max(axis=1).mean())
    return m

def safe_auroc_aupr(binary_labels, score):
    # binary_labels: 0/1
    # score: higher means more likely positive
    out = {"auroc": None, "aupr": None}
    if len(np.unique(binary_labels)) == 2:
        out["auroc"] = float(roc_auc_score(binary_labels, score))
        out["aupr"] = float(average_precision_score(binary_labels, score))
    return out

# -------------------------
# Save/load utilities
# -------------------------
def run_tag(setting: str, split_id: int, seed: int, model_name: str, train_ds: str, test_ds: str):
    return f"{setting}__split{split_id}__seed{seed}__{model_name}__{train_ds}_to_{test_ds}"

def ckpt_path(setting, split_id, seed, model_name, train_ds, test_ds):
    d = CKPT_DIR / setting / f"split_{split_id}" / f"seed_{seed}" / model_name
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{train_ds}_to_{test_ds}.pt"

def pred_path(setting, split_id, seed, model_name, train_ds, test_ds):
    d = PRED_DIR / setting / f"split_{split_id}" / f"seed_{seed}" / model_name
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{train_ds}_to_{test_ds}.npz"

def metr_path(setting, split_id, seed, model_name, train_ds, test_ds):
    d = METR_DIR / setting / f"split_{split_id}" / f"seed_{seed}" / model_name
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{train_ds}_to_{test_ds}.json"

def is_run_done(setting, split_id, seed, model_name, train_ds, test_ds):
    return pred_path(setting, split_id, seed, model_name, train_ds, test_ds).exists() and \
           metr_path(setting, split_id, seed, model_name, train_ds, test_ds).exists()

def save_checkpoint(model, train_info, path: Path, extra: dict):
    payload = {
        "state_dict": model.state_dict(),
        "train_info": train_info,
        "extra": extra,
    }
    torch.save(payload, path)

def save_predictions_npz(path: Path, payload: dict):
    np.savez_compressed(path, **payload)

def save_metrics_json(path: Path, payload: dict):
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)

def cleanup_memory(*objs):
    for o in objs:
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Section A loaded successfully.")
print(f"Experiment root: {EXP_ROOT}")
print(f"Models: {MODELS}")
print(f"Datasets: {DATASETS}")
print(f"Split IDs: {SPLIT_IDS}")
print(f"Seeds: {SEEDS}")

Using DEVICE = cuda
Section A loaded successfully.
Experiment root: /content/drive/MyDrive/PhD Research/iclrFoundationExp/icml_eiml
Models: ['dofa', 'ssl4eo', 'scalemae', 'prithvi', 'sardet100k', 'sarjepa']
Datasets: ['opensarship', 'fusarship']
Split IDs: [0, 1, 2, 3, 4]
Seeds: [42, 123, 456, 789, 999]


### In Domain (ID)

In [ ]:
# @title
# =========================
# Step 2 - Section B (ID runner only)
# =========================
# Runs in-domain experiments:
# - train on dataset d (train split from split_id), validate on d (val), test on d (test)
# - saves checkpoint, per-sample predictions, and metrics
# - resume-safe: skips completed runs

import time
import numpy as np
import torch

# ---------- Optional quick controls ----------
# Set these before running cell if you want smaller test runs:
# RUN_SPLIT_IDS = [0]
# RUN_SEEDS = [42]
# RUN_MODELS = ["dofa"]
# RUN_DATASETS = ["opensarship"]

RUN_SPLIT_IDS = SPLIT_IDS
RUN_SEEDS = SEEDS
RUN_MODELS = MODELS
RUN_DATASETS = DATASETS

SETTING = "ID"

def run_id_one(
    dataset_name: str,
    split_id: int,
    seed: int,
    model_name: str,
):
    train_ds = dataset_name
    test_ds = dataset_name

    # Resume logic
    if is_run_done(SETTING, split_id, seed, model_name, train_ds, test_ds):
        print(f"[SKIP] {run_tag(SETTING, split_id, seed, model_name, train_ds, test_ds)}")
        return

    t0 = time.time()
    seed_everything(seed)

    # 1) Build filtered pool embeddings for this dataset/model
    pool = build_filtered_pool_embeddings(dataset_name, model_name)
    idxs = load_split_indices(dataset_name, split_id)

    tr = slice_by_indices(pool, idxs["train_idx"])
    va = slice_by_indices(pool, idxs["val_idx"])
    te = slice_by_indices(pool, idxs["test_idx"])

    # 2) Dataloaders
    train_loader = make_loader(
        tr["X"], tr["y_id"],
        batch_size=TRAIN_CFG["batch_size"],
        shuffle=True,
        num_workers=TRAIN_CFG["num_workers"],
        pin_memory=TRAIN_CFG["pin_memory"] and (DEVICE == "cuda")
    )
    val_loader = make_loader(
        va["X"], va["y_id"],
        batch_size=TRAIN_CFG["batch_size"],
        shuffle=False,
        num_workers=TRAIN_CFG["num_workers"],
        pin_memory=TRAIN_CFG["pin_memory"] and (DEVICE == "cuda")
    )
    test_loader = make_loader(
        te["X"], te["y_id"],
        batch_size=TRAIN_CFG["batch_size"],
        shuffle=False,
        num_workers=TRAIN_CFG["num_workers"],
        pin_memory=TRAIN_CFG["pin_memory"] and (DEVICE == "cuda")
    )

    # 3) Model + train
    model = MLPHead(
        in_dim=MODEL_DIMS[model_name],
        n_classes=3,
        dropout_p=TRAIN_CFG["dropout_p"]
    ).to(DEVICE)

    model, train_info = train_one_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=DEVICE,
        cfg=TRAIN_CFG
    )

    # Save checkpoint
    ckpt_p = ckpt_path(SETTING, split_id, seed, model_name, train_ds, test_ds)
    save_checkpoint(
        model=model,
        train_info=train_info,
        path=ckpt_p,
        extra={
            "setting": SETTING,
            "split_id": split_id,
            "seed": seed,
            "model_name": model_name,
            "train_dataset": train_ds,
            "test_dataset": test_ds,
            "device": DEVICE,
        }
    )

    # 4) Test predictions
    logits_te, y_true_te = predict_logits(model, test_loader, DEVICE)
    probs_te = softmax_np(logits_te)
    y_pred_te = probs_te.argmax(axis=1).astype(np.int64)
    pmax_te = probs_te.max(axis=1).astype(np.float32)

    # 5) Uncertainty decomposition (prototype-based)
    class_mus, mu_domain = fit_prototypes(tr["X"], tr["y_id"])
    s_class_te, s_domain_te = score_class_domain_uncertainty(te["X"], class_mus, mu_domain)

    # 6) MC-dropout baseline uncertainty
    mc_probs_te, mc_var_te = mc_dropout_probs(model, test_loader, DEVICE, mc_samples=10)
    mc_pmax_te = mc_probs_te.max(axis=1).astype(np.float32)

    # 7) Metrics
    m = compute_basic_metrics(y_true_te, y_pred_te, probs_te)
    m["run_type"] = "ID"
    m["setting"] = SETTING
    m["dataset"] = dataset_name
    m["split_id"] = int(split_id)
    m["seed"] = int(seed)
    m["model_name"] = model_name
    m["n_train"] = int(len(tr["y_id"]))
    m["n_val"] = int(len(va["y_id"]))
    m["n_test"] = int(len(te["y_id"]))
    m["best_epoch"] = int(train_info["best_epoch"])
    m["best_val_loss"] = float(train_info["best_val_loss"])
    m["elapsed_sec"] = float(time.time() - t0)
    m["mean_s_class"] = float(np.mean(s_class_te))
    m["mean_s_domain"] = float(np.mean(s_domain_te))
    m["mean_mc_var"] = float(np.mean(mc_var_te))
    m["mean_mc_pmax"] = float(np.mean(mc_pmax_te))

    # Optional diagnostics: known-vs-error separation in ID
    err_flag = (y_pred_te != y_true_te).astype(np.int32)
    sep_class = safe_auroc_aupr(err_flag, s_class_te)   # higher should indicate error
    sep_domain = safe_auroc_aupr(err_flag, s_domain_te)
    sep_mcvar = safe_auroc_aupr(err_flag, mc_var_te)
    m["error_detection_auroc_aupr"] = {
        "s_class": sep_class,
        "s_domain": sep_domain,
        "mc_var": sep_mcvar,
    }

    # 8) Save per-sample predictions
    pred_p = pred_path(SETTING, split_id, seed, model_name, train_ds, test_ds)
    save_predictions_npz(pred_p, {
        "y_true": y_true_te.astype(np.int16),
        "y_pred": y_pred_te.astype(np.int16),
        "p_max": pmax_te.astype(np.float32),
        "probs": probs_te.astype(np.float32),            # [N,3]
        "mc_p_max": mc_pmax_te.astype(np.float32),
        "mc_var": mc_var_te.astype(np.float32),
        "s_class": s_class_te.astype(np.float32),
        "s_domain": s_domain_te.astype(np.float32),
        "correct": (y_true_te == y_pred_te),
        "is_unknown": np.zeros_like(y_true_te, dtype=bool),  # ID has no unknown class
        "paths": te["paths"].astype(str),
        "label_str_true": np.array([ID_TO_LABEL[int(x)] for x in y_true_te], dtype=object),
        "label_str_pred": np.array([ID_TO_LABEL[int(x)] for x in y_pred_te], dtype=object),
    })

    # 9) Save metrics
    metr_p = metr_path(SETTING, split_id, seed, model_name, train_ds, test_ds)
    save_metrics_json(metr_p, m)

    print(
        f"[DONE] {run_tag(SETTING, split_id, seed, model_name, train_ds, test_ds)} "
        f"| acc={m['accuracy']:.4f} bal_acc={m['balanced_accuracy']:.4f} "
        f"| epoch={m['best_epoch']} | {m['elapsed_sec']:.1f}s"
    )

    # Cleanup
    cleanup_memory(pool, tr, va, te, train_loader, val_loader, test_loader, model)

In [ ]:
def run_id_grid():
    total = 0
    done = 0
    start = time.time()

    for dataset_name in RUN_DATASETS:
        for split_id in RUN_SPLIT_IDS:
            for seed in RUN_SEEDS:
                for model_name in RUN_MODELS:
                    print(dataset_name, ' split: ',split_id, ' seed:', seed, 'model: ', model_name)
                    total += 1
                    try:
                        run_id_one(dataset_name, split_id, seed, model_name)
                        done += 1
                    except Exception as e:
                        print(f"[FAIL] {SETTING} ds={dataset_name} split={split_id} seed={seed} model={model_name} -> {e}")

    elapsed = time.time() - start
    print(f"\nID RUNNER FINISHED: done={done}/{total} | elapsed={elapsed/60:.2f} min")

# Execute
run_id_grid()

opensarship  split:  0  seed: 42 model:  dofa
[SKIP] ID__split0__seed42__dofa__opensarship_to_opensarship
opensarship  split:  0  seed: 42 model:  ssl4eo
[SKIP] ID__split0__seed42__ssl4eo__opensarship_to_opensarship
opensarship  split:  0  seed: 42 model:  scalemae
[SKIP] ID__split0__seed42__scalemae__opensarship_to_opensarship
opensarship  split:  0  seed: 42 model:  prithvi
[SKIP] ID__split0__seed42__prithvi__opensarship_to_opensarship
opensarship  split:  0  seed: 42 model:  sardet100k
[SKIP] ID__split0__seed42__sardet100k__opensarship_to_opensarship
opensarship  split:  0  seed: 42 model:  sarjepa
[SKIP] ID__split0__seed42__sarjepa__opensarship_to_opensarship
opensarship  split:  0  seed: 123 model:  dofa
[SKIP] ID__split0__seed123__dofa__opensarship_to_opensarship
opensarship  split:  0  seed: 123 model:  ssl4eo
[SKIP] ID__split0__seed123__ssl4eo__opensarship_to_opensarship
opensarship  split:  0  seed: 123 model:  scalemae
[SKIP] ID__split0__seed123__scalemae__opensarship_to_open

In [ ]:
# Quick summary for ID results
# - Reads saved JSON metrics
# - Prints mean ± std by (dataset, model)
# - Also prints run coverage so you can see missing runs

import json
import numpy as np
from pathlib import Path
from collections import defaultdict

SETTING = "ID"
METR_ROOT = METR_DIR / SETTING

if not METR_ROOT.exists():
    print(f"No metrics found at: {METR_ROOT}")
else:
    rows = []
    for p in METR_ROOT.rglob("*.json"):
        with open(p, "r") as f:
            m = json.load(f)
        rows.append(m)

    if len(rows) == 0:
        print(f"No ID metric json files found under: {METR_ROOT}")
    else:
        # Group metrics by dataset/model
        grouped = defaultdict(list)
        for r in rows:
            key = (r["dataset"], r["model_name"])
            grouped[key].append(r)

        print(f"Total ID runs found: {len(rows)}\n")

        # Coverage check
        expected = len(RUN_DATASETS) * len(RUN_SPLIT_IDS) * len(RUN_SEEDS) * len(RUN_MODELS)
        print(f"Expected runs (current RUN_* config): {expected}")
        print(f"Found runs: {len(rows)}")
        print(f"Missing: {max(0, expected - len(rows))}\n")

        # Summary table
        header = (
            f"{'Dataset':12s} {'Model':12s} {'n':>3s} "
            f"{'Acc(mean±std)':20s} {'BalAcc(mean±std)':22s} {'MacroF1(mean±std)':22s}"
        )
        print(header)
        print("-" * len(header))

        for dataset_name in sorted(set(k[0] for k in grouped.keys())):
            for model_name in MODELS:
                key = (dataset_name, model_name)
                vals = grouped.get(key, [])
                if len(vals) == 0:
                    print(f"{dataset_name:12s} {model_name:12s} {0:>3d} {'-':20s} {'-':22s} {'-':22s}")
                    continue

                acc = np.array([v["accuracy"] for v in vals], dtype=np.float64)
                bacc = np.array([v["balanced_accuracy"] for v in vals], dtype=np.float64)
                f1m = np.array([v["macro_f1"] for v in vals], dtype=np.float64)

                acc_s = f"{acc.mean():.4f}±{acc.std(ddof=0):.4f}"
                bacc_s = f"{bacc.mean():.4f}±{bacc.std(ddof=0):.4f}"
                f1m_s = f"{f1m.mean():.4f}±{f1m.std(ddof=0):.4f}"

                print(f"{dataset_name:12s} {model_name:12s} {len(vals):>3d} {acc_s:20s} {bacc_s:22s} {f1m_s:22s}")

        # Optional: best model per dataset by mean balanced accuracy
        print("\nBest model per dataset (by mean balanced accuracy):")
        for dataset_name in sorted(set(k[0] for k in grouped.keys())):
            best_model = None
            best_score = -1.0
            for model_name in MODELS:
                vals = grouped.get((dataset_name, model_name), [])
                if not vals:
                    continue
                score = float(np.mean([v["balanced_accuracy"] for v in vals]))
                if score > best_score:
                    best_score = score
                    best_model = model_name
            if best_model is None:
                print(f"- {dataset_name}: no runs")
            else:
                print(f"- {dataset_name}: {best_model} ({best_score:.4f})")

Total ID runs found: 300

Expected runs (current RUN_* config): 300
Found runs: 300
Missing: 0

Dataset      Model          n Acc(mean±std)        BalAcc(mean±std)       MacroF1(mean±std)     
------------------------------------------------------------------------------------------------
fusarship    dofa          25 0.6958±0.0163        0.5657±0.0268          0.5932±0.0206         
fusarship    ssl4eo        25 0.6894±0.0165        0.5751±0.0233          0.6038±0.0241         
fusarship    scalemae      25 0.6649±0.0166        0.5091±0.0475          0.5353±0.0512         
fusarship    prithvi       25 0.6633±0.0181        0.4721±0.0393          0.4958±0.0503         
fusarship    sardet100k    25 0.7568±0.0197        0.7342±0.0398          0.7540±0.0316         
fusarship    sarjepa       25 0.6920±0.0228        0.5834±0.0349          0.6126±0.0330         
opensarship  dofa          25 0.7462±0.0054        0.4438±0.0262          0.4709±0.0341         
opensarship  ssl4eo        25 0

### Cross domain (CD)

In [ ]:
# @title
# =========================
# Step 2 - Section C (CD runner, eval-only from ID checkpoints)
# =========================
# Reuses ID checkpoints:
# - load model trained on source dataset (ID run)
# - evaluate directly on target dataset test split
# - save CD predictions + metrics
#
# This avoids retraining for CD and isolates domain shift cleanly.

import time
import numpy as np
import torch

# ---------- Optional quick controls ----------
# RUN_SPLIT_IDS = [0]
# RUN_SEEDS = [42]
# RUN_MODELS = ["dofa"]

RUN_SPLIT_IDS = SPLIT_IDS
RUN_SEEDS = SEEDS
RUN_MODELS = MODELS

SETTING = "CD"
CD_PAIRS = [("opensarship", "fusarship"), ("fusarship", "opensarship")]

def id_ckpt_path(split_id, seed, model_name, src_dataset):
    # ID checkpoint naming from Section B:
    # checkpoints/ID/split_{i}/seed_{s}/{model}/{src}_to_{src}.pt
    return ckpt_path("ID", split_id, seed, model_name, src_dataset, src_dataset)

def run_cd_one_eval_only(
    train_dataset: str,   # source
    test_dataset: str,    # target
    split_id: int,
    seed: int,
    model_name: str,
):
    if is_run_done(SETTING, split_id, seed, model_name, train_dataset, test_dataset):
        print(f"[SKIP] {run_tag(SETTING, split_id, seed, model_name, train_dataset, test_dataset)}")
        return

    t0 = time.time()
    seed_everything(seed)

    # 1) Load source and target pools
    pool_src = build_filtered_pool_embeddings(train_dataset, model_name)
    pool_tgt = build_filtered_pool_embeddings(test_dataset, model_name)

    idx_src = load_split_indices(train_dataset, split_id)
    idx_tgt = load_split_indices(test_dataset, split_id)

    tr_src = slice_by_indices(pool_src, idx_src["train_idx"])  # for prototypes only
    te_tgt = slice_by_indices(pool_tgt, idx_tgt["test_idx"])   # target test

    # 2) Load ID checkpoint (trained on source domain)
    ckpt_p = id_ckpt_path(split_id, seed, model_name, train_dataset)
    if not ckpt_p.exists():
        raise FileNotFoundError(
            f"Missing ID checkpoint for CD eval-only:\n{ckpt_p}\n"
            f"Run Section B first for source={train_dataset}, split={split_id}, seed={seed}, model={model_name}."
        )

    model = MLPHead(
        in_dim=MODEL_DIMS[model_name],
        n_classes=3,
        dropout_p=TRAIN_CFG["dropout_p"]
    ).to(DEVICE)

    payload = torch.load(ckpt_p, map_location=DEVICE)
    model.load_state_dict(payload["state_dict"])
    id_train_info = payload.get("train_info", {})

    # 3) Test loader on target domain
    test_loader = make_loader(
        te_tgt["X"], te_tgt["y_id"],
        batch_size=TRAIN_CFG["batch_size"],
        shuffle=False,
        num_workers=TRAIN_CFG["num_workers"],
        pin_memory=TRAIN_CFG["pin_memory"] and (DEVICE == "cuda")
    )

    # 4) Predict on target
    logits_te, y_true_te = predict_logits(model, test_loader, DEVICE)
    probs_te = softmax_np(logits_te)
    y_pred_te = probs_te.argmax(axis=1).astype(np.int64)
    pmax_te = probs_te.max(axis=1).astype(np.float32)

    # 5) Uncertainty decomposition
    # prototypes from source TRAIN split (same as ID training domain)
    class_mus_src, mu_domain_src = fit_prototypes(tr_src["X"], tr_src["y_id"])
    s_class_te, s_domain_te = score_class_domain_uncertainty(te_tgt["X"], class_mus_src, mu_domain_src)

    # MC-dropout baseline
    mc_probs_te, mc_var_te = mc_dropout_probs(model, test_loader, DEVICE, mc_samples=10)
    mc_pmax_te = mc_probs_te.max(axis=1).astype(np.float32)

    # 6) Metrics
    m = compute_basic_metrics(y_true_te, y_pred_te, probs_te)
    m["run_type"] = "CD"
    m["setting"] = SETTING
    m["train_dataset"] = train_dataset
    m["test_dataset"] = test_dataset
    m["direction"] = f"{train_dataset}_to_{test_dataset}"
    m["split_id"] = int(split_id)
    m["seed"] = int(seed)
    m["model_name"] = model_name
    m["n_train_source_for_prototypes"] = int(len(tr_src["y_id"]))
    m["n_test_target"] = int(len(te_tgt["y_id"]))
    m["elapsed_sec"] = float(time.time() - t0)

    # copy ID training metadata (for traceability)
    m["id_checkpoint_path"] = str(ckpt_p)
    m["id_best_epoch"] = int(id_train_info.get("best_epoch", -1)) if isinstance(id_train_info, dict) else -1
    m["id_best_val_loss"] = float(id_train_info.get("best_val_loss", np.nan)) if isinstance(id_train_info, dict) else np.nan

    m["mean_s_class"] = float(np.mean(s_class_te))
    m["mean_s_domain"] = float(np.mean(s_domain_te))
    m["mean_mc_var"] = float(np.mean(mc_var_te))
    m["mean_mc_pmax"] = float(np.mean(mc_pmax_te))

    # error detection diagnostics
    err_flag = (y_pred_te != y_true_te).astype(np.int32)
    m["error_detection_auroc_aupr"] = {
        "s_class": safe_auroc_aupr(err_flag, s_class_te),
        "s_domain": safe_auroc_aupr(err_flag, s_domain_te),
        "mc_var": safe_auroc_aupr(err_flag, mc_var_te),
    }

    # 7) Save outputs
    pred_p = pred_path(SETTING, split_id, seed, model_name, train_dataset, test_dataset)
    save_predictions_npz(pred_p, {
        "y_true": y_true_te.astype(np.int16),
        "y_pred": y_pred_te.astype(np.int16),
        "p_max": pmax_te.astype(np.float32),
        "probs": probs_te.astype(np.float32),
        "mc_p_max": mc_pmax_te.astype(np.float32),
        "mc_var": mc_var_te.astype(np.float32),
        "s_class": s_class_te.astype(np.float32),
        "s_domain": s_domain_te.astype(np.float32),
        "correct": (y_true_te == y_pred_te),
        "is_unknown": np.zeros_like(y_true_te, dtype=bool),
        "paths": te_tgt["paths"].astype(str),
        "label_str_true": np.array([ID_TO_LABEL[int(x)] for x in y_true_te], dtype=object),
        "label_str_pred": np.array([ID_TO_LABEL[int(x)] for x in y_pred_te], dtype=object),
    })

    metr_p = metr_path(SETTING, split_id, seed, model_name, train_dataset, test_dataset)
    save_metrics_json(metr_p, m)

    print(
        f"[DONE] {run_tag(SETTING, split_id, seed, model_name, train_dataset, test_dataset)} "
        f"| acc={m['accuracy']:.4f} bal_acc={m['balanced_accuracy']:.4f} "
        f"| (from ID ckpt) | {m['elapsed_sec']:.1f}s"
    )

    cleanup_memory(pool_src, pool_tgt, tr_src, te_tgt, test_loader, model, payload)

In [ ]:
def run_cd_grid_eval_only():
    total, done = 0, 0
    start = time.time()

    for train_dataset, test_dataset in CD_PAIRS:
        for split_id in RUN_SPLIT_IDS:
            for seed in RUN_SEEDS:
                for model_name in RUN_MODELS:
                    total += 1
                    try:
                        run_cd_one_eval_only(train_dataset, test_dataset, split_id, seed, model_name)
                        done += 1
                    except Exception as e:
                        print(
                            f"[FAIL] CD(eval-only) {train_dataset}->{test_dataset} "
                            f"split={split_id} seed={seed} model={model_name} -> {e}"
                        )

    elapsed = time.time() - start
    print(f"\nCD EVAL-ONLY RUNNER FINISHED: done={done}/{total} | elapsed={elapsed/60:.2f} min")

# Execute
run_cd_grid_eval_only()

[SKIP] CD__split0__seed42__dofa__opensarship_to_fusarship
[SKIP] CD__split0__seed42__ssl4eo__opensarship_to_fusarship
[SKIP] CD__split0__seed42__scalemae__opensarship_to_fusarship
[SKIP] CD__split0__seed42__prithvi__opensarship_to_fusarship
[SKIP] CD__split0__seed42__sardet100k__opensarship_to_fusarship
[SKIP] CD__split0__seed42__sarjepa__opensarship_to_fusarship
[SKIP] CD__split0__seed123__dofa__opensarship_to_fusarship
[SKIP] CD__split0__seed123__ssl4eo__opensarship_to_fusarship
[SKIP] CD__split0__seed123__scalemae__opensarship_to_fusarship
[SKIP] CD__split0__seed123__prithvi__opensarship_to_fusarship
[SKIP] CD__split0__seed123__sardet100k__opensarship_to_fusarship
[SKIP] CD__split0__seed123__sarjepa__opensarship_to_fusarship
[SKIP] CD__split0__seed456__dofa__opensarship_to_fusarship
[SKIP] CD__split0__seed456__ssl4eo__opensarship_to_fusarship
[SKIP] CD__split0__seed456__scalemae__opensarship_to_fusarship
[SKIP] CD__split0__seed456__prithvi__opensarship_to_fusarship
[SKIP] CD__split0

In [ ]:
# Quick summary for CD results
# - Reads saved JSON metrics
# - Prints mean ± std by direction and model
# - Shows run coverage and missing runs

import json
import numpy as np
from collections import defaultdict

SETTING = "CD"
METR_ROOT = METR_DIR / SETTING
CD_PAIRS = [("opensarship", "fusarship"), ("fusarship", "opensarship")]

if not METR_ROOT.exists():
    print(f"No metrics found at: {METR_ROOT}")
else:
    rows = []
    for p in METR_ROOT.rglob("*.json"):
        with open(p, "r") as f:
            m = json.load(f)
        rows.append(m)

    if len(rows) == 0:
        print(f"No CD metric json files found under: {METR_ROOT}")
    else:
        print(f"Total CD runs found: {len(rows)}")

        expected = len(CD_PAIRS) * len(RUN_SPLIT_IDS) * len(RUN_SEEDS) * len(RUN_MODELS)
        print(f"Expected runs (current RUN_* config): {expected}")
        print(f"Found runs: {len(rows)}")
        print(f"Missing: {max(0, expected - len(rows))}\n")

        # Group by (direction, model)
        grouped = defaultdict(list)
        for r in rows:
            direction = r.get("direction", f"{r['train_dataset']}_to_{r['test_dataset']}")
            grouped[(direction, r["model_name"])].append(r)

        header = (
            f"{'Direction':28s} {'Model':12s} {'n':>3s} "
            f"{'Acc(mean±std)':20s} {'BalAcc(mean±std)':22s} {'MacroF1(mean±std)':22s}"
        )
        print(header)
        print("-" * len(header))

        all_directions = [f"{a}_to_{b}" for a, b in CD_PAIRS]
        for direction in all_directions:
            for model_name in MODELS:
                vals = grouped.get((direction, model_name), [])
                if len(vals) == 0:
                    print(f"{direction:28s} {model_name:12s} {0:>3d} {'-':20s} {'-':22s} {'-':22s}")
                    continue

                acc = np.array([v["accuracy"] for v in vals], dtype=np.float64)
                bacc = np.array([v["balanced_accuracy"] for v in vals], dtype=np.float64)
                f1m = np.array([v["macro_f1"] for v in vals], dtype=np.float64)

                acc_s = f"{acc.mean():.4f}±{acc.std(ddof=0):.4f}"
                bacc_s = f"{bacc.mean():.4f}±{bacc.std(ddof=0):.4f}"
                f1m_s = f"{f1m.mean():.4f}±{f1m.std(ddof=0):.4f}"

                print(f"{direction:28s} {model_name:12s} {len(vals):>3d} {acc_s:20s} {bacc_s:22s} {f1m_s:22s}")

        # Best model per direction by balanced accuracy
        print("\nBest model per CD direction (by mean balanced accuracy):")
        for direction in all_directions:
            best_model, best_score = None, -1.0
            for model_name in MODELS:
                vals = grouped.get((direction, model_name), [])
                if not vals:
                    continue
                score = float(np.mean([v["balanced_accuracy"] for v in vals]))
                if score > best_score:
                    best_score = score
                    best_model = model_name
            if best_model is None:
                print(f"- {direction}: no runs")
            else:
                print(f"- {direction}: {best_model} ({best_score:.4f})")

Total CD runs found: 300
Expected runs (current RUN_* config): 300
Found runs: 300
Missing: 0

Direction                    Model          n Acc(mean±std)        BalAcc(mean±std)       MacroF1(mean±std)     
----------------------------------------------------------------------------------------------------------------
opensarship_to_fusarship     dofa          25 0.6079±0.0108        0.3158±0.0063          0.2577±0.0074         
opensarship_to_fusarship     ssl4eo        25 0.6382±0.0037        0.3295±0.0018          0.2598±0.0011         
opensarship_to_fusarship     scalemae      25 0.6411±0.0077        0.3310±0.0040          0.2606±0.0019         
opensarship_to_fusarship     prithvi       25 0.6449±0.0032        0.3334±0.0016          0.2630±0.0036         
opensarship_to_fusarship     sardet100k    25 0.5363±0.0398        0.3198±0.0407          0.2597±0.0224         
opensarship_to_fusarship     sarjepa       25 0.5711±0.0295        0.3046±0.0125          0.2518±0.0077         
f

### Unknown class (UC)

In [ ]:
# @title
# =========================
# Step 2 - Section D (UC runner only)
# =========================
# Unknown-class protocol:
# - For each dataset independently
# - Hold out one class as unknown (default: Fishing)
# - Train on remaining known classes (2-class head)
# - Test on full test split (known + unknown)
# - Save checkpoint, per-sample outputs, and UC metrics
# - Resume-safe: skips completed runs

import time
import numpy as np
import torch

# ---------- Optional quick controls ----------
# RUN_SPLIT_IDS = [0]
# RUN_SEEDS = [42]
# RUN_MODELS = ["dofa"]
# RUN_DATASETS = ["opensarship"]
# UNKNOWN_CLASS_NAME = "Fishing"

RUN_SPLIT_IDS = SPLIT_IDS
RUN_SEEDS = SEEDS
RUN_MODELS = MODELS
RUN_DATASETS = DATASETS
UNKNOWN_CLASS_NAME = "Fishing"   # change if needed: "Cargo" or "Tanker"

SETTING = "UC"

def build_uc_subsets(pool, idxs, unknown_class_name):
    """
    Returns:
      train_known, val_known: only known classes (2-class)
      test_full: all three classes (known + unknown)
      meta dict with mapping info
    """
    unknown_id = LABEL_TO_ID[unknown_class_name]
    known_ids = [i for i in [0, 1, 2] if i != unknown_id]
    known_names = [ID_TO_LABEL[k] for k in known_ids]

    tr_full = slice_by_indices(pool, idxs["train_idx"])
    va_full = slice_by_indices(pool, idxs["val_idx"])
    te_full = slice_by_indices(pool, idxs["test_idx"])

    tr_mask_known = np.isin(tr_full["y_id"], np.array(known_ids))
    va_mask_known = np.isin(va_full["y_id"], np.array(known_ids))

    train_known = {
        "X": tr_full["X"][tr_mask_known],
        "y_id_orig": tr_full["y_id"][tr_mask_known],  # original 3-class ids
        "paths": tr_full["paths"][tr_mask_known],
    }
    val_known = {
        "X": va_full["X"][va_mask_known],
        "y_id_orig": va_full["y_id"][va_mask_known],
        "paths": va_full["paths"][va_mask_known],
    }
    test_full = {
        "X": te_full["X"],
        "y_id_orig": te_full["y_id"],                 # includes unknown
        "paths": te_full["paths"],
    }

    # remap known original ids -> [0,1] for training head
    orig_to_known2 = {known_ids[0]: 0, known_ids[1]: 1}
    known2_to_orig = {0: known_ids[0], 1: known_ids[1]}

    y_tr_2 = np.array([orig_to_known2[int(v)] for v in train_known["y_id_orig"]], dtype=np.int64)
    y_va_2 = np.array([orig_to_known2[int(v)] for v in val_known["y_id_orig"]], dtype=np.int64)

    # test known mask
    te_is_unknown = (test_full["y_id_orig"] == unknown_id)
    te_is_known = ~te_is_unknown

    # safety checks
    if len(np.unique(y_tr_2)) < 2:
        raise ValueError("UC train set has <2 known classes after filtering.")
    if len(np.unique(y_va_2)) < 2:
        raise ValueError("UC val set has <2 known classes after filtering.")

    meta = {
        "unknown_class_name": unknown_class_name,
        "unknown_class_id_orig": int(unknown_id),
        "known_class_ids_orig": [int(x) for x in known_ids],
        "known_class_names": known_names,
        "orig_to_known2": {str(k): int(v) for k, v in orig_to_known2.items()},
        "known2_to_orig": {str(k): int(v) for k, v in known2_to_orig.items()},
    }

    return train_known, y_tr_2, val_known, y_va_2, test_full, te_is_known, te_is_unknown, meta

def fit_known_prototypes(train_known_X, y_train_2):
    # Two known classes in 2-class space [0,1]
    mu_domain = train_known_X.mean(axis=0, keepdims=True)
    class_mus = []
    for c in [0, 1]:
        Xc = train_known_X[y_train_2 == c]
        if len(Xc) == 0:
            raise ValueError(f"No samples in known class {c} for prototype fitting.")
        class_mus.append(Xc.mean(axis=0))
    class_mus = np.stack(class_mus, axis=0).astype(np.float32)  # [2,D]
    return class_mus, mu_domain.astype(np.float32)

def run_uc_one(
    dataset_name: str,
    split_id: int,
    seed: int,
    model_name: str,
    unknown_class_name: str = "Fishing",
):
    train_ds = dataset_name
    test_ds = dataset_name

    # Use test_ds field with suffix to keep unique file naming per unknown class setting
    test_ds_tag = f"{test_ds}_unknown_{unknown_class_name}"

    if is_run_done(SETTING, split_id, seed, model_name, train_ds, test_ds_tag):
        print(f"[SKIP] {run_tag(SETTING, split_id, seed, model_name, train_ds, test_ds_tag)}")
        return

    t0 = time.time()
    seed_everything(seed)

    # 1) Pool + split
    pool = build_filtered_pool_embeddings(dataset_name, model_name)
    idxs = load_split_indices(dataset_name, split_id)

    (
        tr_known, y_tr_2,
        va_known, y_va_2,
        te_full, te_is_known, te_is_unknown,
        uc_meta
    ) = build_uc_subsets(pool, idxs, unknown_class_name=unknown_class_name)

    # 2) Loaders (2-class training)
    train_loader = make_loader(
        tr_known["X"], y_tr_2,
        batch_size=TRAIN_CFG["batch_size"],
        shuffle=True,
        num_workers=TRAIN_CFG["num_workers"],
        pin_memory=TRAIN_CFG["pin_memory"] and (DEVICE == "cuda")
    )
    val_loader = make_loader(
        va_known["X"], y_va_2,
        batch_size=TRAIN_CFG["batch_size"],
        shuffle=False,
        num_workers=TRAIN_CFG["num_workers"],
        pin_memory=TRAIN_CFG["pin_memory"] and (DEVICE == "cuda")
    )

    # For test forward pass we use known dummy labels (not used), size must match
    y_dummy_test = np.zeros(len(te_full["X"]), dtype=np.int64)
    test_loader = make_loader(
        te_full["X"], y_dummy_test,
        batch_size=TRAIN_CFG["batch_size"],
        shuffle=False,
        num_workers=TRAIN_CFG["num_workers"],
        pin_memory=TRAIN_CFG["pin_memory"] and (DEVICE == "cuda")
    )

    # 3) 2-class model head
    model = MLPHead(
        in_dim=MODEL_DIMS[model_name],
        n_classes=2,
        dropout_p=TRAIN_CFG["dropout_p"]
    ).to(DEVICE)

    model, train_info = train_one_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=DEVICE,
        cfg=TRAIN_CFG
    )

    # Save checkpoint
    ckpt_p = ckpt_path(SETTING, split_id, seed, model_name, train_ds, test_ds_tag)
    save_checkpoint(
        model=model,
        train_info=train_info,
        path=ckpt_p,
        extra={
            "setting": SETTING,
            "split_id": split_id,
            "seed": seed,
            "model_name": model_name,
            "train_dataset": train_ds,
            "test_dataset": test_ds,
            "test_dataset_tag": test_ds_tag,
            "device": DEVICE,
            "uc_meta": uc_meta,
        }
    )

    # 4) Predict on full test (known + unknown)
    logits_te_2, _ = predict_logits(model, test_loader, DEVICE)     # [N,2]
    probs_te_2 = softmax_np(logits_te_2)                             # [N,2]
    pred_known2 = probs_te_2.argmax(axis=1).astype(np.int64)         # [N]
    pmax_te_2 = probs_te_2.max(axis=1).astype(np.float32)

    # Map 2-class predictions back to original 3-class ids for known predictions
    known2_to_orig = {int(k): int(v) for k, v in uc_meta["known2_to_orig"].items()}
    y_pred_orig = np.array([known2_to_orig[int(k)] for k in pred_known2], dtype=np.int64)

    # 5) Uncertainty scores
    # Prototype scores from known training only
    class_mus_2, mu_domain = fit_known_prototypes(tr_known["X"], y_tr_2)
    s_class_te, s_domain_te = score_class_domain_uncertainty(te_full["X"], class_mus_2, mu_domain)

    # MC-dropout
    mc_probs_te_2, mc_var_te = mc_dropout_probs(model, test_loader, DEVICE, mc_samples=10)
    mc_pmax_te_2 = mc_probs_te_2.max(axis=1).astype(np.float32)

    # 6) UC detection targets and metrics
    # Unknown detection: positive class = unknown
    y_unknown_bin = te_is_unknown.astype(np.int32)

    # Candidate uncertainty scores (higher => more uncertain/unknown)
    # - 1 - pmax (higher means less confident)
    # - mc_var
    # - s_class
    # - s_domain
    u_softmax = (1.0 - pmax_te_2).astype(np.float32)
    u_mc = mc_var_te.astype(np.float32)
    u_class = s_class_te.astype(np.float32)
    u_domain = s_domain_te.astype(np.float32)

    uc_softmax = safe_auroc_aupr(y_unknown_bin, u_softmax)
    uc_mc = safe_auroc_aupr(y_unknown_bin, u_mc)
    uc_class = safe_auroc_aupr(y_unknown_bin, u_class)
    uc_domain = safe_auroc_aupr(y_unknown_bin, u_domain)

    # Known-only classification quality (on known test samples only)
    known_mask = te_is_known
    y_true_known_orig = te_full["y_id_orig"][known_mask]
    y_pred_known_orig = y_pred_orig[known_mask]

    known_acc = float((y_true_known_orig == y_pred_known_orig).mean()) if known_mask.sum() > 0 else np.nan

    # Balanced acc among known classes only
    # remap known original ids to {0,1} for balanced accuracy
    orig_to_known2 = {int(k): int(v) for k, v in uc_meta["orig_to_known2"].items()}
    y_true_known2 = np.array([orig_to_known2[int(v)] for v in y_true_known_orig], dtype=np.int64)
    y_pred_known2 = np.array([orig_to_known2.get(int(v), -1) for v in y_pred_known_orig], dtype=np.int64)
    # y_pred_known_orig should always be known, but guard anyway
    valid = (y_pred_known2 >= 0)
    if valid.sum() > 0 and len(np.unique(y_true_known2[valid])) > 1:
        from sklearn.metrics import balanced_accuracy_score
        known_bal_acc = float(balanced_accuracy_score(y_true_known2[valid], y_pred_known2[valid]))
    else:
        known_bal_acc = np.nan

    # 7) Aggregate metrics
    m = {
        "run_type": "UC",
        "setting": SETTING,
        "dataset": dataset_name,
        "split_id": int(split_id),
        "seed": int(seed),
        "model_name": model_name,
        "unknown_class_name": unknown_class_name,
        "unknown_class_id_orig": int(uc_meta["unknown_class_id_orig"]),
        "known_class_ids_orig": uc_meta["known_class_ids_orig"],
        "known_class_names": uc_meta["known_class_names"],
        "n_train_known": int(len(y_tr_2)),
        "n_val_known": int(len(y_va_2)),
        "n_test_total": int(len(te_full["y_id_orig"])),
        "n_test_known": int(known_mask.sum()),
        "n_test_unknown": int(te_is_unknown.sum()),
        "best_epoch": int(train_info["best_epoch"]),
        "best_val_loss": float(train_info["best_val_loss"]),
        "elapsed_sec": float(time.time() - t0),
        "known_only_accuracy": known_acc,
        "known_only_balanced_accuracy": known_bal_acc,
        "uc_detection": {
            "softmax_1_minus_pmax": uc_softmax,
            "mc_var": uc_mc,
            "s_class": uc_class,
            "s_domain": uc_domain,
        },
        "score_means": {
            "mean_1_minus_pmax": float(u_softmax.mean()),
            "mean_mc_var": float(u_mc.mean()),
            "mean_s_class": float(u_class.mean()),
            "mean_s_domain": float(u_domain.mean()),
        }
    }

    # 8) Save predictions
    # Keep both original-3-class truth and known2 outputs for analysis flexibility
    pred_p = pred_path(SETTING, split_id, seed, model_name, train_ds, test_ds_tag)
    save_predictions_npz(pred_p, {
        "y_true_orig3": te_full["y_id_orig"].astype(np.int16),         # 0..2
        "y_pred_orig3": y_pred_orig.astype(np.int16),                  # predicts known classes only
        "is_unknown": te_is_unknown.astype(bool),
        "is_known": te_is_known.astype(bool),

        "probs_known2": probs_te_2.astype(np.float32),                 # [N,2]
        "pmax_known2": pmax_te_2.astype(np.float32),
        "u_softmax_1_minus_pmax": u_softmax.astype(np.float32),
        "u_mc_var": u_mc.astype(np.float32),
        "u_s_class": u_class.astype(np.float32),
        "u_s_domain": u_domain.astype(np.float32),

        "mc_probs_known2": mc_probs_te_2.astype(np.float32),           # [N,2]
        "mc_pmax_known2": mc_pmax_te_2.astype(np.float32),

        "paths": te_full["paths"].astype(str),
        "label_str_true_orig3": np.array([ID_TO_LABEL[int(x)] for x in te_full["y_id_orig"]], dtype=object),
        "label_str_pred_orig3": np.array([ID_TO_LABEL[int(x)] for x in y_pred_orig], dtype=object),
    })

    # 9) Save metrics
    metr_p = metr_path(SETTING, split_id, seed, model_name, train_ds, test_ds_tag)
    save_metrics_json(metr_p, m)

    print(
        f"[DONE] {run_tag(SETTING, split_id, seed, model_name, train_ds, test_ds_tag)} "
        f"| known_acc={m['known_only_accuracy']:.4f} "
        f"| UC(s_class auroc)={m['uc_detection']['s_class']['auroc'] if m['uc_detection']['s_class']['auroc'] is not None else 'NA'} "
        f"| epoch={m['best_epoch']} | {m['elapsed_sec']:.1f}s"
    )

    cleanup_memory(pool, tr_known, va_known, te_full, train_loader, val_loader, test_loader, model)

In [ ]:
# @title
def run_uc_grid():
    total, done = 0, 0
    start = time.time()

    for dataset_name in RUN_DATASETS:
        for split_id in RUN_SPLIT_IDS:
            for seed in RUN_SEEDS:
                for model_name in RUN_MODELS:
                    print(f"[{SETTING}] ds={dataset_name} split={split_id} seed={seed} model={model_name}")
                    total += 1
                    try:
                        run_uc_one(
                            dataset_name=dataset_name,
                            split_id=split_id,
                            seed=seed,
                            model_name=model_name,
                            unknown_class_name=UNKNOWN_CLASS_NAME
                        )
                        done += 1
                    except Exception as e:
                        print(
                            f"[FAIL] {SETTING} ds={dataset_name} split={split_id} "
                            f"seed={seed} model={model_name} unknown={UNKNOWN_CLASS_NAME} -> {e}"
                        )

    elapsed = time.time() - start
    print(f"\nUC RUNNER FINISHED: done={done}/{total} | elapsed={elapsed/60:.2f} min")

# Execute
run_uc_grid()

[UC] ds=opensarship split=0 seed=42 model=dofa
[SKIP] UC__split0__seed42__dofa__opensarship_to_opensarship_unknown_Fishing
[UC] ds=opensarship split=0 seed=42 model=ssl4eo
[SKIP] UC__split0__seed42__ssl4eo__opensarship_to_opensarship_unknown_Fishing
[UC] ds=opensarship split=0 seed=42 model=scalemae
[SKIP] UC__split0__seed42__scalemae__opensarship_to_opensarship_unknown_Fishing
[UC] ds=opensarship split=0 seed=42 model=prithvi
[SKIP] UC__split0__seed42__prithvi__opensarship_to_opensarship_unknown_Fishing
[UC] ds=opensarship split=0 seed=42 model=sardet100k
[SKIP] UC__split0__seed42__sardet100k__opensarship_to_opensarship_unknown_Fishing
[UC] ds=opensarship split=0 seed=42 model=sarjepa
[SKIP] UC__split0__seed42__sarjepa__opensarship_to_opensarship_unknown_Fishing
[UC] ds=opensarship split=0 seed=123 model=dofa
[SKIP] UC__split0__seed123__dofa__opensarship_to_opensarship_unknown_Fishing
[UC] ds=opensarship split=0 seed=123 model=ssl4eo
[SKIP] UC__split0__seed123__ssl4eo__opensarship_to_

In [ ]:
UNKNOWN_CLASS_NAME = "Cargo"   # change if needed: "Cargo" or "Tanker"

def run_uc_grid():
    total, done = 0, 0
    start = time.time()

    for dataset_name in RUN_DATASETS:
        for split_id in RUN_SPLIT_IDS:
            for seed in RUN_SEEDS:
                for model_name in RUN_MODELS:
                    print(f"[{SETTING}] ds={dataset_name} split={split_id} seed={seed} model={model_name}")
                    total += 1
                    try:
                        run_uc_one(
                            dataset_name=dataset_name,
                            split_id=split_id,
                            seed=seed,
                            model_name=model_name,
                            unknown_class_name=UNKNOWN_CLASS_NAME
                        )
                        done += 1
                    except Exception as e:
                        print(
                            f"[FAIL] {SETTING} ds={dataset_name} split={split_id} "
                            f"seed={seed} model={model_name} unknown={UNKNOWN_CLASS_NAME} -> {e}"
                        )

    elapsed = time.time() - start
    print(f"\nUC RUNNER FINISHED: done={done}/{total} | elapsed={elapsed/60:.2f} min")

# Execute
run_uc_grid()

[UC] ds=opensarship split=0 seed=42 model=dofa
[SKIP] UC__split0__seed42__dofa__opensarship_to_opensarship_unknown_Cargo
[UC] ds=opensarship split=0 seed=42 model=ssl4eo
[SKIP] UC__split0__seed42__ssl4eo__opensarship_to_opensarship_unknown_Cargo
[UC] ds=opensarship split=0 seed=42 model=scalemae
[SKIP] UC__split0__seed42__scalemae__opensarship_to_opensarship_unknown_Cargo
[UC] ds=opensarship split=0 seed=42 model=prithvi
[SKIP] UC__split0__seed42__prithvi__opensarship_to_opensarship_unknown_Cargo
[UC] ds=opensarship split=0 seed=42 model=sardet100k
[SKIP] UC__split0__seed42__sardet100k__opensarship_to_opensarship_unknown_Cargo
[UC] ds=opensarship split=0 seed=42 model=sarjepa
[SKIP] UC__split0__seed42__sarjepa__opensarship_to_opensarship_unknown_Cargo
[UC] ds=opensarship split=0 seed=123 model=dofa
[SKIP] UC__split0__seed123__dofa__opensarship_to_opensarship_unknown_Cargo
[UC] ds=opensarship split=0 seed=123 model=ssl4eo
[SKIP] UC__split0__seed123__ssl4eo__opensarship_to_opensarship_un

In [ ]:
UNKNOWN_CLASS_NAME = "Tanker"   # change if needed: "Cargo" or "Tanker" or "Fishing"

def run_uc_grid():
    total, done = 0, 0
    start = time.time()

    for dataset_name in RUN_DATASETS:
        for split_id in RUN_SPLIT_IDS:
            for seed in RUN_SEEDS:
                for model_name in RUN_MODELS:
                    print(f"[{SETTING}] ds={dataset_name} split={split_id} seed={seed} model={model_name}")
                    total += 1
                    try:
                        run_uc_one(
                            dataset_name=dataset_name,
                            split_id=split_id,
                            seed=seed,
                            model_name=model_name,
                            unknown_class_name=UNKNOWN_CLASS_NAME
                        )
                        done += 1
                    except Exception as e:
                        print(
                            f"[FAIL] {SETTING} ds={dataset_name} split={split_id} "
                            f"seed={seed} model={model_name} unknown={UNKNOWN_CLASS_NAME} -> {e}"
                        )

    elapsed = time.time() - start
    print(f"\nUC RUNNER FINISHED: done={done}/{total} | elapsed={elapsed/60:.2f} min")

# Execute
run_uc_grid()

[UC] ds=opensarship split=0 seed=42 model=dofa
[SKIP] UC__split0__seed42__dofa__opensarship_to_opensarship_unknown_Tanker
[UC] ds=opensarship split=0 seed=42 model=ssl4eo
[SKIP] UC__split0__seed42__ssl4eo__opensarship_to_opensarship_unknown_Tanker
[UC] ds=opensarship split=0 seed=42 model=scalemae
[SKIP] UC__split0__seed42__scalemae__opensarship_to_opensarship_unknown_Tanker
[UC] ds=opensarship split=0 seed=42 model=prithvi
[SKIP] UC__split0__seed42__prithvi__opensarship_to_opensarship_unknown_Tanker
[UC] ds=opensarship split=0 seed=42 model=sardet100k
[SKIP] UC__split0__seed42__sardet100k__opensarship_to_opensarship_unknown_Tanker
[UC] ds=opensarship split=0 seed=42 model=sarjepa
[SKIP] UC__split0__seed42__sarjepa__opensarship_to_opensarship_unknown_Tanker
[UC] ds=opensarship split=0 seed=123 model=dofa
[SKIP] UC__split0__seed123__dofa__opensarship_to_opensarship_unknown_Tanker
[UC] ds=opensarship split=0 seed=123 model=ssl4eo
[SKIP] UC__split0__seed123__ssl4eo__opensarship_to_opensar

In [ ]:
# Quick summary for UC results
# - Reads saved JSON metrics
# - Prints mean ± std by dataset/model for selected unknown class
# - Shows run coverage

import json
import numpy as np
from collections import defaultdict

SETTING = "UC"
METR_ROOT = METR_DIR / SETTING

# should match what you used in runner
UNKNOWN_CLASS_NAME = "Fishing"

if not METR_ROOT.exists():
    print(f"No metrics found at: {METR_ROOT}")
else:
    rows = []
    for p in METR_ROOT.rglob("*.json"):
        with open(p, "r") as f:
            m = json.load(f)
        if m.get("unknown_class_name", None) == UNKNOWN_CLASS_NAME:
            rows.append(m)

    if len(rows) == 0:
        print(f"No UC metric json files found for unknown={UNKNOWN_CLASS_NAME} under: {METR_ROOT}")
    else:
        print(f"Total UC runs found (unknown={UNKNOWN_CLASS_NAME}): {len(rows)}")

        expected = len(RUN_DATASETS) * len(RUN_SPLIT_IDS) * len(RUN_SEEDS) * len(RUN_MODELS)
        print(f"Expected runs (current RUN_* config): {expected}")
        print(f"Found runs: {len(rows)}")
        print(f"Missing: {max(0, expected - len(rows))}\n")

        def get_nested(dct, keys, default=np.nan):
            cur = dct
            for k in keys:
                if not isinstance(cur, dict) or k not in cur:
                    return default
                cur = cur[k]
            if cur is None:
                return default
            return float(cur)

        grouped = defaultdict(list)
        for r in rows:
            key = (r["dataset"], r["model_name"])
            grouped[key].append(r)

        header = (
            f"{'Dataset':12s} {'Model':12s} {'n':>3s} "
            f"{'KnownAcc(mean±std)':22s} "
            f"{'AUROC s_class':14s} {'AUROC s_domain':15s} {'AUROC mc_var':14s} {'AUROC 1-pmax':14s}"
        )
        print(header)
        print("-" * len(header))

        for dataset_name in RUN_DATASETS:
            for model_name in MODELS:
                vals = grouped.get((dataset_name, model_name), [])
                if len(vals) == 0:
                    print(f"{dataset_name:12s} {model_name:12s} {0:>3d} {'-':22s} {'-':14s} {'-':15s} {'-':14s} {'-':14s}")
                    continue

                known_acc = np.array([get_nested(v, ["known_only_accuracy"]) for v in vals], dtype=np.float64)
                au_sclass = np.array([get_nested(v, ["uc_detection", "s_class", "auroc"]) for v in vals], dtype=np.float64)
                au_sdomain = np.array([get_nested(v, ["uc_detection", "s_domain", "auroc"]) for v in vals], dtype=np.float64)
                au_mc = np.array([get_nested(v, ["uc_detection", "mc_var", "auroc"]) for v in vals], dtype=np.float64)
                au_soft = np.array([get_nested(v, ["uc_detection", "softmax_1_minus_pmax", "auroc"]) for v in vals], dtype=np.float64)

                def fmt(arr):
                    arr = arr[~np.isnan(arr)]
                    if len(arr) == 0:
                        return "-"
                    return f"{arr.mean():.4f}±{arr.std(ddof=0):.4f}"

                print(
                    f"{dataset_name:12s} {model_name:12s} {len(vals):>3d} "
                    f"{fmt(known_acc):22s} "
                    f"{fmt(au_sclass):14s} {fmt(au_sdomain):15s} {fmt(au_mc):14s} {fmt(au_soft):14s}"
                )

        # Best model by AUROC(s_class), per dataset
        print(f"\nBest model per dataset by UC AUROC(s_class), unknown={UNKNOWN_CLASS_NAME}:")
        for dataset_name in RUN_DATASETS:
            best_model = None
            best_score = -1.0
            for model_name in MODELS:
                vals = grouped.get((dataset_name, model_name), [])
                if not vals:
                    continue
                arr = np.array([get_nested(v, ["uc_detection", "s_class", "auroc"]) for v in vals], dtype=np.float64)
                arr = arr[~np.isnan(arr)]
                if len(arr) == 0:
                    continue
                score = float(arr.mean())
                if score > best_score:
                    best_score = score
                    best_model = model_name
            if best_model is None:
                print(f"- {dataset_name}: no valid runs")
            else:
                print(f"- {dataset_name}: {best_model} ({best_score:.4f})")

KeyboardInterrupt: 

## Analysis

In [ ]:
# =========================
# Step 3: Full analysis + paper-ready outputs
# =========================
# This script:
# 1) Loads saved Step 2 metrics/predictions from icml_eiml
# 2) Builds aggregated mean±std tables for ID, CD, UC
# 3) Creates ranking tables (paper-ready CSV)
# 4) Saves per-run flat tables for traceability
# 5) Saves compact markdown summaries for direct paper drafting
#
# No training, CPU only.

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

assert "OUTPUT_ROOT" in globals(), "Please define OUTPUT_ROOT as pathlib.Path first."
OUTPUT_ROOT = Path(OUTPUT_ROOT)

# -------------------------
# Paths
# -------------------------
EXP_ROOT = OUTPUT_ROOT / "icml_eiml"
METR_DIR = EXP_ROOT / "metrics"
PRED_DIR = EXP_ROOT / "predictions"
AN_DIR = EXP_ROOT / "analysis"
FINAL_DIR = AN_DIR / "final_tables"
FIGDATA_DIR = AN_DIR / "figure_data"
RUNLOG_DIR = AN_DIR / "run_logs"

for d in [AN_DIR, FINAL_DIR, FIGDATA_DIR, RUNLOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# -------------------------
# Config
# -------------------------
MODELS = ["dofa", "ssl4eo", "scalemae", "prithvi", "sardet100k", "sarjepa"]
DATASETS = ["opensarship", "fusarship"]
# UC_UNKNOWN_CLASS = "Fishing"   # must match your UC runner setting
UC_UNKNOWN_CLASSES = ["Fishing", "Cargo", "Tanker"]   # add/remove as needed


# -------------------------
# Helpers
# -------------------------
def load_all_metric_json(setting: str):
    root = METR_DIR / setting
    rows = []
    if not root.exists():
        return rows
    for p in root.rglob("*.json"):
        try:
            with open(p, "r") as f:
                obj = json.load(f)
            obj["_metric_file"] = str(p)
            rows.append(obj)
        except Exception as e:
            print(f"[WARN] Failed to load {p}: {e}")
    return rows

def to_float_or_nan(x):
    if x is None:
        return np.nan
    try:
        return float(x)
    except:
        return np.nan

def mean_std_str(x):
    x = np.asarray(x, dtype=np.float64)
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return "-"
    return f"{x.mean():.4f} ± {x.std(ddof=0):.4f}"

def flatten_classwise_accuracy(classwise_obj):
    out = {}
    if isinstance(classwise_obj, dict):
        for k, v in classwise_obj.items():
            out[f"class_acc_{k}"] = to_float_or_nan(v)
    return out

def risk_coverage_from_predictions(y_true, y_pred, uncertainty, coverages=np.linspace(0.1, 1.0, 10)):
    """
    uncertainty: higher = more uncertain
    keep lowest-uncertainty samples to achieve coverage
    returns list of dicts with coverage and risk(error rate)
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    u = np.asarray(uncertainty, dtype=np.float64)
    n = len(y_true)
    if n == 0:
        return []

    order = np.argsort(u)  # least uncertain first
    out = []
    for c in coverages:
        k = max(1, int(round(c * n)))
        idx = order[:k]
        err = (y_pred[idx] != y_true[idx]).mean()
        out.append({"coverage": float(k / n), "risk": float(err)})
    return out

def save_md_table(df: pd.DataFrame, path: Path, title: str):
    with open(path, "w") as f:
        f.write(f"# {title}\n\n")
        if len(df) == 0:
            f.write("No data.\n")
        else:
            f.write(df.to_markdown(index=False))
            f.write("\n")

# -------------------------
# Load metrics by setting
# -------------------------
id_rows = load_all_metric_json("ID")
cd_rows = load_all_metric_json("CD")
uc_rows = load_all_metric_json("UC")

print(f"Loaded metrics: ID={len(id_rows)}, CD={len(cd_rows)}, UC={len(uc_rows)}")

# -------------------------
# Build flat run-level dataframes
# -------------------------
# ID
id_records = []
for r in id_rows:
    rec = {
        "setting": "ID",
        "dataset": r.get("dataset", ""),
        "model_name": r.get("model_name", ""),
        "split_id": r.get("split_id", np.nan),
        "seed": r.get("seed", np.nan),
        "accuracy": to_float_or_nan(r.get("accuracy")),
        "balanced_accuracy": to_float_or_nan(r.get("balanced_accuracy")),
        "macro_f1": to_float_or_nan(r.get("macro_f1")),
        "best_epoch": to_float_or_nan(r.get("best_epoch")),
        "best_val_loss": to_float_or_nan(r.get("best_val_loss")),
        "elapsed_sec": to_float_or_nan(r.get("elapsed_sec")),
        "mean_s_class": to_float_or_nan(r.get("mean_s_class")),
        "mean_s_domain": to_float_or_nan(r.get("mean_s_domain")),
        "mean_mc_var": to_float_or_nan(r.get("mean_mc_var")),
        "mean_mc_pmax": to_float_or_nan(r.get("mean_mc_pmax")),
        "_metric_file": r.get("_metric_file", ""),
    }
    rec.update(flatten_classwise_accuracy(r.get("classwise_accuracy", {})))
    # Error detection
    ed = r.get("error_detection_auroc_aupr", {})
    rec["errdet_s_class_auroc"] = to_float_or_nan(ed.get("s_class", {}).get("auroc") if isinstance(ed.get("s_class", {}), dict) else np.nan)
    rec["errdet_s_domain_auroc"] = to_float_or_nan(ed.get("s_domain", {}).get("auroc") if isinstance(ed.get("s_domain", {}), dict) else np.nan)
    rec["errdet_mc_var_auroc"] = to_float_or_nan(ed.get("mc_var", {}).get("auroc") if isinstance(ed.get("mc_var", {}), dict) else np.nan)
    id_records.append(rec)

df_id = pd.DataFrame(id_records)

# CD
cd_records = []
for r in cd_rows:
    direction = r.get("direction", f"{r.get('train_dataset','')}_to_{r.get('test_dataset','')}")
    rec = {
        "setting": "CD",
        "direction": direction,
        "train_dataset": r.get("train_dataset", ""),
        "test_dataset": r.get("test_dataset", ""),
        "model_name": r.get("model_name", ""),
        "split_id": r.get("split_id", np.nan),
        "seed": r.get("seed", np.nan),
        "accuracy": to_float_or_nan(r.get("accuracy")),
        "balanced_accuracy": to_float_or_nan(r.get("balanced_accuracy")),
        "macro_f1": to_float_or_nan(r.get("macro_f1")),
        "best_epoch": to_float_or_nan(r.get("best_epoch")),
        "best_val_loss": to_float_or_nan(r.get("best_val_loss")),
        "elapsed_sec": to_float_or_nan(r.get("elapsed_sec")),
        "mean_s_class": to_float_or_nan(r.get("mean_s_class")),
        "mean_s_domain": to_float_or_nan(r.get("mean_s_domain")),
        "mean_mc_var": to_float_or_nan(r.get("mean_mc_var")),
        "mean_mc_pmax": to_float_or_nan(r.get("mean_mc_pmax")),
        "_metric_file": r.get("_metric_file", ""),
    }
    rec.update(flatten_classwise_accuracy(r.get("classwise_accuracy", {})))
    ed = r.get("error_detection_auroc_aupr", {})
    rec["errdet_s_class_auroc"] = to_float_or_nan(ed.get("s_class", {}).get("auroc") if isinstance(ed.get("s_class", {}), dict) else np.nan)
    rec["errdet_s_domain_auroc"] = to_float_or_nan(ed.get("s_domain", {}).get("auroc") if isinstance(ed.get("s_domain", {}), dict) else np.nan)
    rec["errdet_mc_var_auroc"] = to_float_or_nan(ed.get("mc_var", {}).get("auroc") if isinstance(ed.get("mc_var", {}), dict) else np.nan)
    cd_records.append(rec)

df_cd = pd.DataFrame(cd_records)

# UC
# uc_records = []
# for r in uc_rows:
#     if r.get("unknown_class_name", None) != UC_UNKNOWN_CLASS:
#         continue
#     ucd = r.get("uc_detection", {})
#     rec = {
#         "setting": "UC",
#         "dataset": r.get("dataset", ""),
#         "model_name": r.get("model_name", ""),
#         "split_id": r.get("split_id", np.nan),
#         "seed": r.get("seed", np.nan),
#         "unknown_class_name": r.get("unknown_class_name", ""),
#         "known_only_accuracy": to_float_or_nan(r.get("known_only_accuracy")),
#         "known_only_balanced_accuracy": to_float_or_nan(r.get("known_only_balanced_accuracy")),
#         "best_epoch": to_float_or_nan(r.get("best_epoch")),
#         "best_val_loss": to_float_or_nan(r.get("best_val_loss")),
#         "elapsed_sec": to_float_or_nan(r.get("elapsed_sec")),

#         # UC detection AUROC/AUPR for each score
#         "uc_softmax_auroc": to_float_or_nan(ucd.get("softmax_1_minus_pmax", {}).get("auroc") if isinstance(ucd.get("softmax_1_minus_pmax", {}), dict) else np.nan),
#         "uc_softmax_aupr": to_float_or_nan(ucd.get("softmax_1_minus_pmax", {}).get("aupr") if isinstance(ucd.get("softmax_1_minus_pmax", {}), dict) else np.nan),
#         "uc_mc_var_auroc": to_float_or_nan(ucd.get("mc_var", {}).get("auroc") if isinstance(ucd.get("mc_var", {}), dict) else np.nan),
#         "uc_mc_var_aupr": to_float_or_nan(ucd.get("mc_var", {}).get("aupr") if isinstance(ucd.get("mc_var", {}), dict) else np.nan),
#         "uc_s_class_auroc": to_float_or_nan(ucd.get("s_class", {}).get("auroc") if isinstance(ucd.get("s_class", {}), dict) else np.nan),
#         "uc_s_class_aupr": to_float_or_nan(ucd.get("s_class", {}).get("aupr") if isinstance(ucd.get("s_class", {}), dict) else np.nan),
#         "uc_s_domain_auroc": to_float_or_nan(ucd.get("s_domain", {}).get("auroc") if isinstance(ucd.get("s_domain", {}), dict) else np.nan),
#         "uc_s_domain_aupr": to_float_or_nan(ucd.get("s_domain", {}).get("aupr") if isinstance(ucd.get("s_domain", {}), dict) else np.nan),

#         "_metric_file": r.get("_metric_file", ""),
#     }
#     uc_records.append(rec)

# df_uc = pd.DataFrame(uc_records)

# -------------------------
# UC
# -------------------------
uc_records = []
for r in uc_rows:
    unk = r.get("unknown_class_name", None)
    if unk not in UC_UNKNOWN_CLASSES:
        continue

    ucd = r.get("uc_detection", {})
    rec = {
        "setting": "UC",
        "dataset": r.get("dataset", ""),
        "model_name": r.get("model_name", ""),
        "split_id": r.get("split_id", np.nan),
        "seed": r.get("seed", np.nan),
        "unknown_class_name": unk,
        "known_only_accuracy": to_float_or_nan(r.get("known_only_accuracy")),
        "known_only_balanced_accuracy": to_float_or_nan(r.get("known_only_balanced_accuracy")),
        "best_epoch": to_float_or_nan(r.get("best_epoch")),
        "best_val_loss": to_float_or_nan(r.get("best_val_loss")),
        "elapsed_sec": to_float_or_nan(r.get("elapsed_sec")),

        "uc_softmax_auroc": to_float_or_nan(ucd.get("softmax_1_minus_pmax", {}).get("auroc") if isinstance(ucd.get("softmax_1_minus_pmax", {}), dict) else np.nan),
        "uc_softmax_aupr":  to_float_or_nan(ucd.get("softmax_1_minus_pmax", {}).get("aupr")  if isinstance(ucd.get("softmax_1_minus_pmax", {}), dict) else np.nan),
        "uc_mc_var_auroc":  to_float_or_nan(ucd.get("mc_var", {}).get("auroc") if isinstance(ucd.get("mc_var", {}), dict) else np.nan),
        "uc_mc_var_aupr":   to_float_or_nan(ucd.get("mc_var", {}).get("aupr")  if isinstance(ucd.get("mc_var", {}), dict) else np.nan),
        "uc_s_class_auroc": to_float_or_nan(ucd.get("s_class", {}).get("auroc") if isinstance(ucd.get("s_class", {}), dict) else np.nan),
        "uc_s_class_aupr":  to_float_or_nan(ucd.get("s_class", {}).get("aupr")  if isinstance(ucd.get("s_class", {}), dict) else np.nan),
        "uc_s_domain_auroc": to_float_or_nan(ucd.get("s_domain", {}).get("auroc") if isinstance(ucd.get("s_domain", {}), dict) else np.nan),
        "uc_s_domain_aupr":  to_float_or_nan(ucd.get("s_domain", {}).get("aupr")  if isinstance(ucd.get("s_domain", {}), dict) else np.nan),

        "_metric_file": r.get("_metric_file", ""),
    }
    uc_records.append(rec)

df_uc = pd.DataFrame(uc_records)

# Save all UC raw runs
df_uc.to_csv(RUNLOG_DIR / "uc_runs_raw_all_unknowns.csv", index=False)

# Save per-unknown raw runs
for unk in UC_UNKNOWN_CLASSES:
    sub = df_uc[df_uc["unknown_class_name"] == unk].copy()
    sub.to_csv(RUNLOG_DIR / f"uc_runs_raw_unknown_{unk.lower()}.csv", index=False)

# Save raw run logs
df_id.to_csv(RUNLOG_DIR / "id_runs_raw.csv", index=False)
df_cd.to_csv(RUNLOG_DIR / "cd_runs_raw.csv", index=False)
# df_uc.to_csv(RUNLOG_DIR / "uc_runs_raw.csv", index=False)

print(f"Saved raw run logs to: {RUNLOG_DIR}")

# -------------------------
# Aggregation helpers
# -------------------------
def aggregate_mean_std(df, group_cols, metric_cols):
    if len(df) == 0:
        return pd.DataFrame()

    agg_rows = []
    grp = df.groupby(group_cols, dropna=False)
    for keys, sub in grp:
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {c: k for c, k in zip(group_cols, keys)}
        row["n_runs"] = len(sub)
        for m in metric_cols:
            vals = pd.to_numeric(sub[m], errors="coerce").dropna().values
            row[f"{m}_mean"] = float(np.mean(vals)) if len(vals) else np.nan
            row[f"{m}_std"] = float(np.std(vals, ddof=0)) if len(vals) else np.nan
            row[f"{m}_mean_std"] = mean_std_str(vals)
        agg_rows.append(row)
    return pd.DataFrame(agg_rows)

# -------------------------
# ID aggregated tables
# -------------------------
id_metric_cols = [
    "accuracy", "balanced_accuracy", "macro_f1",
    "class_acc_Cargo", "class_acc_Tanker", "class_acc_Fishing",
    "errdet_s_class_auroc", "errdet_s_domain_auroc", "errdet_mc_var_auroc"
]

id_agg = aggregate_mean_std(df_id, ["dataset", "model_name"], id_metric_cols)
if len(id_agg):
    id_agg = id_agg.sort_values(["dataset", "balanced_accuracy_mean"], ascending=[True, False]).reset_index(drop=True)

id_agg.to_csv(FINAL_DIR / "ID_aggregated_mean_std.csv", index=False)
save_md_table(
    id_agg[[
        "dataset", "model_name", "n_runs",
        "accuracy_mean_std", "balanced_accuracy_mean_std", "macro_f1_mean_std",
        "class_acc_Cargo_mean_std", "class_acc_Tanker_mean_std", "class_acc_Fishing_mean_std"
    ]] if len(id_agg) else id_agg,
    FINAL_DIR / "ID_aggregated_mean_std.md",
    "ID Results (mean ± std)"
)

# ID ranking per dataset
id_rank_rows = []
for ds in DATASETS:
    sub = id_agg[id_agg["dataset"] == ds].copy() if len(id_agg) else pd.DataFrame()
    if len(sub) == 0:
        continue
    sub = sub.sort_values("balanced_accuracy_mean", ascending=False).reset_index(drop=True)
    sub["rank_bal_acc"] = np.arange(1, len(sub) + 1)
    id_rank_rows.append(sub[["dataset", "rank_bal_acc", "model_name", "balanced_accuracy_mean_std", "accuracy_mean_std", "macro_f1_mean_std"]])
id_rank = pd.concat(id_rank_rows, axis=0) if len(id_rank_rows) else pd.DataFrame()
id_rank.to_csv(FINAL_DIR / "ID_ranking_by_balanced_accuracy.csv", index=False)

# -------------------------
# CD aggregated tables
# -------------------------
cd_metric_cols = [
    "accuracy", "balanced_accuracy", "macro_f1",
    "class_acc_Cargo", "class_acc_Tanker", "class_acc_Fishing",
    "errdet_s_class_auroc", "errdet_s_domain_auroc", "errdet_mc_var_auroc"
]

cd_agg = aggregate_mean_std(df_cd, ["direction", "model_name"], cd_metric_cols)
if len(cd_agg):
    cd_agg = cd_agg.sort_values(["direction", "balanced_accuracy_mean"], ascending=[True, False]).reset_index(drop=True)

cd_agg.to_csv(FINAL_DIR / "CD_aggregated_mean_std.csv", index=False)
save_md_table(
    cd_agg[[
        "direction", "model_name", "n_runs",
        "accuracy_mean_std", "balanced_accuracy_mean_std", "macro_f1_mean_std",
        "class_acc_Cargo_mean_std", "class_acc_Tanker_mean_std", "class_acc_Fishing_mean_std"
    ]] if len(cd_agg) else cd_agg,
    FINAL_DIR / "CD_aggregated_mean_std.md",
    "CD Results (mean ± std)"
)

# CD ranking per direction
cd_rank_rows = []
if len(cd_agg):
    for direction in sorted(cd_agg["direction"].unique()):
        sub = cd_agg[cd_agg["direction"] == direction].copy()
        sub = sub.sort_values("balanced_accuracy_mean", ascending=False).reset_index(drop=True)
        sub["rank_bal_acc"] = np.arange(1, len(sub) + 1)
        cd_rank_rows.append(sub[["direction", "rank_bal_acc", "model_name", "balanced_accuracy_mean_std", "accuracy_mean_std", "macro_f1_mean_std"]])
cd_rank = pd.concat(cd_rank_rows, axis=0) if len(cd_rank_rows) else pd.DataFrame()
cd_rank.to_csv(FINAL_DIR / "CD_ranking_by_balanced_accuracy.csv", index=False)

# -------------------------
# UC aggregated tables
# -------------------------
# uc_metric_cols = [
#     "known_only_accuracy", "known_only_balanced_accuracy",
#     "uc_softmax_auroc", "uc_softmax_aupr",
#     "uc_mc_var_auroc", "uc_mc_var_aupr",
#     "uc_s_class_auroc", "uc_s_class_aupr",
#     "uc_s_domain_auroc", "uc_s_domain_aupr",
# ]

# uc_agg = aggregate_mean_std(df_uc, ["dataset", "unknown_class_name", "model_name"], uc_metric_cols)
# if len(uc_agg):
#     uc_agg = uc_agg.sort_values(["dataset", "uc_s_class_auroc_mean"], ascending=[True, False]).reset_index(drop=True)

# uc_agg.to_csv(FINAL_DIR / "UC_aggregated_mean_std.csv", index=False)
# save_md_table(
#     uc_agg[[
#         "dataset", "unknown_class_name", "model_name", "n_runs",
#         "known_only_accuracy_mean_std", "known_only_balanced_accuracy_mean_std",
#         "uc_s_class_auroc_mean_std", "uc_s_domain_auroc_mean_std", "uc_mc_var_auroc_mean_std", "uc_softmax_auroc_mean_std"
#     ]] if len(uc_agg) else uc_agg,
#     FINAL_DIR / "UC_aggregated_mean_std.md",
#     f"UC Results (mean ± std), unknown={UC_UNKNOWN_CLASS}"
# )

uc_metric_cols = [
    "known_only_accuracy", "known_only_balanced_accuracy",
    "uc_softmax_auroc", "uc_softmax_aupr",
    "uc_mc_var_auroc", "uc_mc_var_aupr",
    "uc_s_class_auroc", "uc_s_class_aupr",
    "uc_s_domain_auroc", "uc_s_domain_aupr",
]

uc_agg = aggregate_mean_std(df_uc, ["dataset", "unknown_class_name", "model_name"], uc_metric_cols)
if len(uc_agg):
    uc_agg = uc_agg.sort_values(
        ["unknown_class_name", "dataset", "uc_s_class_auroc_mean"],
        ascending=[True, True, False]
    ).reset_index(drop=True)

# all-in-one
uc_agg.to_csv(FINAL_DIR / "UC_aggregated_mean_std_all_unknowns.csv", index=False)

# per-unknown files
for unk in UC_UNKNOWN_CLASSES:
    sub = uc_agg[uc_agg["unknown_class_name"] == unk].copy()
    sub.to_csv(FINAL_DIR / f"UC_aggregated_mean_std_unknown_{unk.lower()}.csv", index=False)
    save_md_table(
        sub[[
            "dataset", "unknown_class_name", "model_name", "n_runs",
            "known_only_accuracy_mean_std", "known_only_balanced_accuracy_mean_std",
            "uc_s_class_auroc_mean_std", "uc_s_domain_auroc_mean_std",
            "uc_mc_var_auroc_mean_std", "uc_softmax_auroc_mean_std"
        ]] if len(sub) else sub,
        FINAL_DIR / f"UC_aggregated_mean_std_unknown_{unk.lower()}.md",
        f"UC Results (mean ± std), unknown={unk}"
    )

# # UC ranking by s_class AUROC
# uc_rank_rows = []
# if len(uc_agg):
#     for ds in DATASETS:
#         sub = uc_agg[(uc_agg["dataset"] == ds) & (uc_agg["unknown_class_name"] == UC_UNKNOWN_CLASS)].copy()
#         if len(sub) == 0:
#             continue
#         sub = sub.sort_values("uc_s_class_auroc_mean", ascending=False).reset_index(drop=True)
#         sub["rank_uc_s_class_auroc"] = np.arange(1, len(sub) + 1)
#         uc_rank_rows.append(sub[[
#             "dataset", "unknown_class_name", "rank_uc_s_class_auroc", "model_name",
#             "uc_s_class_auroc_mean_std", "uc_s_domain_auroc_mean_std", "uc_mc_var_auroc_mean_std", "uc_softmax_auroc_mean_std"
#         ]])
# uc_rank = pd.concat(uc_rank_rows, axis=0) if len(uc_rank_rows) else pd.DataFrame()
# uc_rank.to_csv(FINAL_DIR / "UC_ranking_by_s_class_auroc.csv", index=False)



# -------------------------
# Combined paper summary table
# -------------------------
# One row per model, with compact columns:
# - ID bal-acc averaged across datasets
# - CD bal-acc averaged across directions
# - UC s_class AUROC averaged across datasets
paper_rows = []
for model in MODELS:
    row = {"model_name": model}

    # ID
    if len(id_agg):
        sub = id_agg[id_agg["model_name"] == model]
        row["ID_bal_acc_mean_over_datasets"] = to_float_or_nan(sub["balanced_accuracy_mean"].mean() if len(sub) else np.nan)
        row["ID_bal_acc_std_over_datasets"] = to_float_or_nan(sub["balanced_accuracy_mean"].std(ddof=0) if len(sub) else np.nan)
    else:
        row["ID_bal_acc_mean_over_datasets"] = np.nan
        row["ID_bal_acc_std_over_datasets"] = np.nan

    # CD
    if len(cd_agg):
        sub = cd_agg[cd_agg["model_name"] == model]
        row["CD_bal_acc_mean_over_directions"] = to_float_or_nan(sub["balanced_accuracy_mean"].mean() if len(sub) else np.nan)
        row["CD_bal_acc_std_over_directions"] = to_float_or_nan(sub["balanced_accuracy_mean"].std(ddof=0) if len(sub) else np.nan)
    else:
        row["CD_bal_acc_mean_over_directions"] = np.nan
        row["CD_bal_acc_std_over_directions"] = np.nan

    # # UC
    # if len(uc_agg):
    #     sub = uc_agg[(uc_agg["model_name"] == model) & (uc_agg["unknown_class_name"] == UC_UNKNOWN_CLASS)]
    #     row["UC_s_class_auroc_mean_over_datasets"] = to_float_or_nan(sub["uc_s_class_auroc_mean"].mean() if len(sub) else np.nan)
    #     row["UC_s_class_auroc_std_over_datasets"] = to_float_or_nan(sub["uc_s_class_auroc_mean"].std(ddof=0) if len(sub) else np.nan)
    # else:
    #     row["UC_s_class_auroc_mean_over_datasets"] = np.nan
    #     row["UC_s_class_auroc_std_over_datasets"] = np.nan
    # UC (aggregate over all selected unknown classes and datasets)

    if len(uc_agg):
        sub = uc_agg[
            (uc_agg["model_name"] == model) &
            (uc_agg["unknown_class_name"].isin(UC_UNKNOWN_CLASSES))
        ]

        row["UC_s_class_auroc_mean_over_datasets"] = to_float_or_nan(
            sub["uc_s_class_auroc_mean"].mean() if len(sub) else np.nan
        )
        row["UC_s_class_auroc_std_over_datasets"] = to_float_or_nan(
            sub["uc_s_class_auroc_mean"].std(ddof=0) if len(sub) else np.nan
        )
    else:
        row["UC_s_class_auroc_mean_over_datasets"] = np.nan
        row["UC_s_class_auroc_std_over_datasets"] = np.nan

    # Composite score (optional, equal weight)
    comps = [
        row["ID_bal_acc_mean_over_datasets"],
        row["CD_bal_acc_mean_over_directions"],
        row["UC_s_class_auroc_mean_over_datasets"],
    ]
    comps = np.array(comps, dtype=np.float64)
    row["composite_mean"] = float(np.nanmean(comps)) if np.any(~np.isnan(comps)) else np.nan

    paper_rows.append(row)

df_paper = pd.DataFrame(paper_rows)
df_paper = df_paper.sort_values("composite_mean", ascending=False).reset_index(drop=True)
df_paper["rank_composite"] = np.arange(1, len(df_paper) + 1)

df_paper.to_csv(FINAL_DIR / "PAPER_combined_model_summary.csv", index=False)
save_md_table(
    df_paper[[
        "rank_composite", "model_name",
        "ID_bal_acc_mean_over_datasets",
        "CD_bal_acc_mean_over_directions",
        "UC_s_class_auroc_mean_over_datasets",
        "composite_mean"
    ]],
    FINAL_DIR / "PAPER_combined_model_summary.md",
    "Combined Model Summary (ID, CD, UC)"
)

# -------------------------
# Figure-ready data: risk-coverage using saved predictions
# -------------------------
# We create compact CSVs per setting/model for later plotting.
def collect_prediction_files(setting):
    root = PRED_DIR / setting
    if not root.exists():
        return []
    return list(root.rglob("*.npz"))

# ID/CD risk-coverage on correctness
for setting in ["ID", "CD"]:
    files = collect_prediction_files(setting)
    rc_rows = []
    for fp in files:
        try:
            z = np.load(fp, allow_pickle=False)
            y_true = z["y_true"]
            y_pred = z["y_pred"]
            # uncertainty candidates
            u_class = z["s_class"]
            u_domain = z["s_domain"]
            u_mc = z["mc_var"] if "mc_var" in z.files else None
            u_soft = 1.0 - z["p_max"]

            base = {
                "setting": setting,
                "file": str(fp),
            }

            for name, u in [("s_class", u_class), ("s_domain", u_domain), ("mc_var", u_mc), ("softmax_1mpmax", u_soft)]:
                if u is None:
                    continue
                rc = risk_coverage_from_predictions(y_true, y_pred, u)
                for r in rc:
                    rr = base.copy()
                    rr["uncertainty"] = name
                    rr["coverage"] = r["coverage"]
                    rr["risk"] = r["risk"]
                    rc_rows.append(rr)
        except Exception as e:
            print(f"[WARN] risk-coverage failed for {fp}: {e}")

    df_rc = pd.DataFrame(rc_rows)
    df_rc.to_csv(FIGDATA_DIR / f"{setting}_risk_coverage_points.csv", index=False)

# UC figure data: unknown detection score distributions summary
uc_files = collect_prediction_files("UC")
uc_fig_rows = []
for fp in uc_files:
    try:
        z = np.load(fp, allow_pickle=False)
        is_unknown = z["is_unknown"].astype(bool)

        for key in ["u_s_class", "u_s_domain", "u_mc_var", "u_softmax_1_minus_pmax"]:
            if key not in z.files:
                continue
            v = z[key].astype(np.float64)
            kn = v[~is_unknown]
            un = v[is_unknown]
            uc_fig_rows.append({
                "file": str(fp),
                "score": key,
                "known_mean": float(np.mean(kn)) if len(kn) else np.nan,
                "known_std": float(np.std(kn, ddof=0)) if len(kn) else np.nan,
                "unknown_mean": float(np.mean(un)) if len(un) else np.nan,
                "unknown_std": float(np.std(un, ddof=0)) if len(un) else np.nan,
                "known_n": int(len(kn)),
                "unknown_n": int(len(un)),
            })
    except Exception as e:
        print(f"[WARN] UC figure data failed for {fp}: {e}")

df_uc_fig = pd.DataFrame(uc_fig_rows)
df_uc_fig.to_csv(FIGDATA_DIR / "UC_score_distribution_summary.csv", index=False)

# -------------------------
# Coverage and sanity report
# -------------------------
def expected_count_id():
    # 2 datasets * 5 splits * 5 seeds * 6 models
    return len(DATASETS) * 5 * 5 * len(MODELS)

def expected_count_cd():
    # 2 directions * 5 splits * 5 seeds * 6 models
    return 2 * 5 * 5 * len(MODELS)

# def expected_count_uc():
#     # 2 datasets * 5 splits * 5 seeds * 6 models (for one unknown class)
#     return len(DATASETS) * 5 * 5 * len(MODELS)

def expected_count_uc_per_unknown():
    return len(DATASETS) * 5 * 5 * len(MODELS)

coverage = {
    "ID_found": int(len(df_id)),
    "ID_expected_nominal": int(expected_count_id()),
    "CD_found": int(len(df_cd)),
    "CD_expected_nominal": int(expected_count_cd()),
    "UC_found_total_selected_unknowns": int(len(df_uc)),
    "UC_expected_total_selected_unknowns": int(expected_count_uc_per_unknown() * len(UC_UNKNOWN_CLASSES)),
}
for unk in UC_UNKNOWN_CLASSES:
    coverage[f"UC_found_unknown_{unk}"] = int((df_uc["unknown_class_name"] == unk).sum()) if len(df_uc) else 0
    coverage[f"UC_expected_unknown_{unk}"] = int(expected_count_uc_per_unknown())

# coverage = {
#     "ID_found": int(len(df_id)),
#     "ID_expected_nominal": int(expected_count_id()),
#     "CD_found": int(len(df_cd)),
#     "CD_expected_nominal": int(expected_count_cd()),
#     "UC_found_unknown_"+UC_UNKNOWN_CLASS: int(len(df_uc)),
#     "UC_expected_nominal": int(expected_count_uc()),
# }
with open(AN_DIR / "coverage_report.json", "w") as f:
    json.dump(coverage, f, indent=2)

print("\n==============================")
print("Step 3 analysis complete.")
print(f"Final tables: {FINAL_DIR}")
print(f"Figure data: {FIGDATA_DIR}")
print(f"Raw run logs: {RUNLOG_DIR}")
print(f"Coverage report: {AN_DIR / 'coverage_report.json'}")
print("==============================")

print("\nCoverage:")
print(json.dumps(coverage, indent=2))

print("\nTop combined models:")
display(df_paper.head(10))

Loaded metrics: ID=300, CD=300, UC=900
Saved raw run logs to: /content/drive/MyDrive/PhD Research/iclrFoundationExp/icml_eiml/analysis/run_logs

Step 3 analysis complete.
Final tables: /content/drive/MyDrive/PhD Research/iclrFoundationExp/icml_eiml/analysis/final_tables
Figure data: /content/drive/MyDrive/PhD Research/iclrFoundationExp/icml_eiml/analysis/figure_data
Raw run logs: /content/drive/MyDrive/PhD Research/iclrFoundationExp/icml_eiml/analysis/run_logs
Coverage report: /content/drive/MyDrive/PhD Research/iclrFoundationExp/icml_eiml/analysis/coverage_report.json

Coverage:
{
  "ID_found": 300,
  "ID_expected_nominal": 300,
  "CD_found": 300,
  "CD_expected_nominal": 300,
  "UC_found_total_selected_unknowns": 900,
  "UC_expected_total_selected_unknowns": 900,
  "UC_found_unknown_Fishing": 300,
  "UC_expected_unknown_Fishing": 300,
  "UC_found_unknown_Cargo": 300,
  "UC_expected_unknown_Cargo": 300,
  "UC_found_unknown_Tanker": 300,
  "UC_expected_unknown_Tanker": 300
}

Top combi

,model_name,ID_bal_acc_mean_over_datasets,ID_bal_acc_std_over_datasets,CD_bal_acc_mean_over_directions,CD_bal_acc_std_over_directions,UC_s_class_auroc_mean_over_datasets,UC_s_class_auroc_std_over_datasets,composite_mean,rank_composite
0,sardet100k,0.633221,0.101003,0.343955,0.024158,0.602336,0.119047,0.526504,1
1,dofa,0.504762,0.060983,0.386044,0.070267,0.589440,0.117248,0.493415,2
2,ssl4eo,0.478359,0.096705,0.363276,0.033790,0.606164,0.115075,0.482599,3
3,sarjepa,0.519719,0.063712,0.298549,0.006033,0.624017,0.140311,0.480762,4
4,prithvi,0.415789,0.056278,0.383709,0.050308,0.632282,0.138669,0.477260,5
5,scalemae,0.448019,0.061113,0.319042,0.011937,0.630904,0.145347,0.465988,6


In [ ]:
import inspect, os, glob
from pathlib import Path

OUT_DIR = OUTPUT_ROOT
print("OUT_DIR =", OUT_DIR)

for fn_name in ["pred_path", "metr_path", "run_tag"]:
    fn = globals().get(fn_name, None)
    print("\n---", fn_name, "---")
    print("found:", fn is not None)
    if fn is not None:
        try:
            print(inspect.getsource(fn))
        except OSError:
            print("Could not print source (maybe defined in a notebook cell). Try printing repr:", fn)

# list likely saved artifacts under OUT_DIR
out = Path(OUT_DIR)
print("\nExisting .npz under OUT_DIR (first 30):")
npzs = sorted(out.rglob("*.npz"))
print("count:", len(npzs))
for p in npzs[:30]:
    print(p)

print("\nExisting .json under OUT_DIR (first 30):")
jsons = sorted(out.rglob("*.json"))
print("count:", len(jsons))
for p in jsons[:30]:
    print(p)

OUT_DIR = /content/drive/MyDrive/PhD Research/iclrFoundationExp

--- pred_path ---
found: True
def pred_path(setting, split_id, seed, model_name, train_ds, test_ds):
    d = PRED_DIR / setting / f"split_{split_id}" / f"seed_{seed}" / model_name
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{train_ds}_to_{test_ds}.npz"


--- metr_path ---
found: True
def metr_path(setting, split_id, seed, model_name, train_ds, test_ds):
    d = METR_DIR / setting / f"split_{split_id}" / f"seed_{seed}" / model_name
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{train_ds}_to_{test_ds}.json"


--- run_tag ---
found: True
def run_tag(setting: str, split_id: int, seed: int, model_name: str, train_ds: str, test_ds: str):
    return f"{setting}__split{split_id}__seed{seed}__{model_name}__{train_ds}_to_{test_ds}"


Existing .npz under OUT_DIR (first 30):
count: 1611
/content/drive/MyDrive/PhD Research/iclrFoundationExp/checkpoints/dinov3/fusarship_test_checkpoint.npz
/content/drive/MyDri

KeyboardInterrupt: 

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

setting="CD"
split_id=0
seed=42
model_name="prithvi"
train_ds="opensarship"
test_ds="fusarship"

p = pred_path(setting, split_id, seed, model_name, train_ds, test_ds)
print("NPZ:", p)

z = np.load(p, allow_pickle=True)
y_true = z["y_true"]
y_pred = z["y_pred"]

print("y_true counts:", np.bincount(y_true))
print("y_pred counts:", np.bincount(y_pred))

cm = confusion_matrix(y_true, y_pred, labels=[0,1,2])
print("confusion matrix rows=true cols=pred:\n", cm)
print("keys:", list(z.keys()))

NPZ: /content/drive/MyDrive/PhD Research/iclrFoundationExp/icml_eiml/predictions/CD/split_0/seed_42/prithvi/opensarship_to_fusarship.npz
y_true counts: [170  15  78]
y_pred counts: [263]
confusion matrix rows=true cols=pred:
 [[170   0   0]
 [ 15   0   0]
 [ 78   0   0]]
keys: ['y_true', 'y_pred', 'p_max', 'probs', 'mc_p_max', 'mc_var', 's_class', 's_domain', 'correct', 'is_unknown', 'paths', 'label_str_true', 'label_str_pred']


In [ ]:
# discussion_risk_map.py
# Creates a single-column friendly colorful discussion chart and saves as PDF.

import matplotlib.pyplot as plt

# ---------- Style ----------
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "pdf.fonttype": 42,   # editable text in vector PDF
    "ps.fonttype": 42
})

# Single-column width (~3.4 inches for many venues)
fig, ax = plt.subplots(figsize=(3.4, 2.6), dpi=300)

# ---------- Background risk zones ----------
ax.axvspan(0, 0.45, ymin=0.0, ymax=0.45, color="#f8d7da", alpha=0.45, zorder=0)  # high risk
ax.axvspan(0.45, 1.0, ymin=0.0, ymax=0.45, color="#fff3cd", alpha=0.45, zorder=0) # caution
ax.axvspan(0, 0.45, ymin=0.45, ymax=1.0, color="#d1ecf1", alpha=0.45, zorder=0)   # monitor
ax.axvspan(0.45, 1.0, ymin=0.45, ymax=1.0, color="#d4edda", alpha=0.45, zorder=0) # reliable

# ---------- Points (conceptual, non-result) ----------
# x: classification reliability, y: uncertainty reliability
points = {
    "In-domain": (0.78, 0.72, "#2ca02c"),
    "Cross-domain": (0.32, 0.38, "#d62728"),
    "Unknown-class\n(favorable)": (0.58, 0.74, "#1f77b4"),
    "Unknown-class\n(challenging)": (0.42, 0.47, "#ff7f0e"),
}

for label, (x, y, c) in points.items():
    ax.scatter(x, y, s=42, color=c, edgecolor="black", linewidth=0.5, zorder=3)

# ---------- Label offsets to avoid overlap ----------
offsets = {
    "In-domain": (6, 6),
    "Cross-domain": (6, -10),
    "Unknown-class\n(favorable)": (6, 6),
    "Unknown-class\n(challenging)": (-62, -12),
}

for label, (x, y, c) in points.items():
    dx, dy = offsets[label]
    ax.annotate(
        label, (x, y),
        xytext=(dx, dy), textcoords="offset points",
        ha="left" if dx >= 0 else "right",
        va="bottom" if dy >= 0 else "top",
        fontsize=7,
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.9),
        zorder=4
    )

# ---------- Axes ----------
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Classification reliability")
ax.set_ylabel("Uncertainty reliability")
ax.set_title("Discussion risk map", pad=4)

ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.5)
for spine in ax.spines.values():
    spine.set_linewidth(0.8)

# Quadrant guide lines
ax.axvline(0.45, color="gray", linestyle=":", linewidth=0.8)
ax.axhline(0.45, color="gray", linestyle=":", linewidth=0.8)

# Tight layout to avoid clipping
fig.tight_layout(pad=0.5)

# Save PDF
fig.savefig("discussion_risk_map.pdf", bbox_inches="tight")
plt.close(fig)
print("Saved: discussion_risk_map.pdf")

Saved: discussion_risk_map.pdf


In [ ]:
# decision_matrix_figure.py
# Creates a single-column decision matrix figure (table-like) and saves as PDF.

import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 7.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

# Single-column friendly size
fig, ax = plt.subplots(figsize=(3.4, 2.8), dpi=300)
ax.axis("off")

columns = [
    "Scenario",
    "Primary report",
    "Uncertainty use",
    "Action"
]

rows = [
    ["In-domain", "Balanced Acc, Macro-F1", "Optional monitoring", "Deploy with routine checks"],
    ["Cross-domain", "Balanced Acc, per-class recall", "Required for triage", "Gate high-uncertainty cases"],
    ["Unknown-class risk", "AUROC/AUPR + known recall", "Required for reject option", "Reject/flag uncertain samples"],
    ["Pre-deployment check", "Bidirectional CD + multi-UC", "Compare score types", "Approve only if stable"]
]

# Create table
tbl = ax.table(
    cellText=rows,
    colLabels=columns,
    loc="center",
    cellLoc="left",
    colLoc="left"
)

# Styling
tbl.auto_set_font_size(False)
tbl.set_fontsize(7)
tbl.scale(1.0, 1.35)

# Header and row colors
header_color = "#2f3e4e"
row_colors = ["#f7f9fc", "#eef3f9"]

for (r, c), cell in tbl.get_celld().items():
    cell.set_linewidth(0.6)
    if r == 0:
        cell.set_facecolor(header_color)
        cell.set_text_props(color="white", weight="bold")
    else:
        cell.set_facecolor(row_colors[(r - 1) % 2])

# Improve column widths to reduce overlap/clipping
# (index: width fraction in axes coordinates)
col_widths = {0: 0.23, 1: 0.27, 2: 0.22, 3: 0.28}
for (r, c), cell in tbl.get_celld().items():
    if c in col_widths:
        cell.set_width(col_widths[c])

fig.tight_layout(pad=0.4)
fig.savefig("decision_matrix.pdf", bbox_inches="tight")
plt.close(fig)
print("Saved: decision_matrix.pdf")

Saved: decision_matrix.pdf
